#Environment Setup

In [ ]:
# !pip install openai
# !pip install bert-score
# !pip install transformers
# !pip install huggingface
# !pip install dataset
# !pip install accelerate
# !pip install --upgrade typing_extensions
# !pip install openai
# !pip install --upgrade transformers
# !pip install datasets
# !pip install sentence-transformers
# !pip install scikit-learn
# !pip install bert-score rouge-score nltk
# !pip install tqdm
# !pip install -q transformers datasets detoxify accelerate
# !pip install -U spacy
# !python -m spacy download en_core_web_sm
# !pip install textstat
# !pip install bitsandbytes
# !pip install tabulate

# Login into HuggingFace

To use the expert and amateur models referenced in the paper, you must first authenticate with HuggingFace. These models are gated and require access approval.

**Required Models**

  * **Amateur Models**: [TinyLlama-1.1B-Chat-v1.0](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0), [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)

  * **Expert Models**: [Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)

# Dataset Loading -> [1k_dataset](https://drive.google.com/file/d/1YHlMR4-3XwrSlX-S5ohnkITL3wyLaFXZ/view?usp=sharing)

**Constructing 1k_dataset code** --> [1k_dataset code](https://colab.research.google.com/drive/1CZWdXo-KIMU1bXTHKu3p06qTjYrj8eTy?usp=sharing)

**Sample Dataset Entry**

**Prompt:**  
`who are the basques and where do they live`

**Original Answer:**  
> The Basques (/bɑːsks/ or /bæsks/; Basque: euskaldunak; Spanish: vascos; French: basques) are an indigenous ethnic group characterized by the Basque language, a common culture, and shared ancestry to the ancient Vascones and Aquitanians.  
> They primarily inhabit the Basque Country (Euskal Herria), a region around the western end of the Pyrenees, straddling parts of north-central Spain and southwestern France.

**Score:** `1.0`

**Feedback:**  
> The answer is accurate, complete, and well-written. It provides a detailed explanation of who the Basques are...  

**Improved Answer:**  
> The Basques are an indigenous ethnic group primarily living in the Basque Country... They are characterized by their unique language, Basque, and a shared ancestry tracing back to the ancient Vascones and Aquitanians.


In [ ]:
import json

def generate_training_dataset(dataset_name):
    file_path = dataset_name
    expert_datasets = []

    with open(file_path, "r") as f:
        for line in f:
            item = json.loads(line)

            prompt = item["prompt"]
            original_answer = item["original_answer"]
            score = item["score"]
            feedback = item["feedback"]
            expert_datasets.append(
                {"prompt": prompt,
                "answer": original_answer,
                "feedback": feedback,
                "score": score}
        )

    print(f" {len(expert_datasets)} examples loaded")
    return expert_datasets

In [ ]:
is_math = True
dataset_path = "300_sample.jsonl" if is_math else "1k_dataset.jsonl"
expert_datasets = generate_training_dataset(dataset_path)

# Model Loading

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import torch

teacher_model_name = "meta-llama/Llama-3.1-8B-Instruct"
student_model_name = "meta-llama/Llama-3.2-1B-Instruct"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_model_name, trust_remote_code=True)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_name,
    device_map = "auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

student_tokenizer = AutoTokenizer.from_pretrained(student_model_name, trust_remote_code=True)
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

nli_model_name = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_model_name,
    device_map="auto"
)

##Few Shot Prompting

In [ ]:
math_few_shot_examples = [
    {
        "question": "A train travels at 80 km/h for 3 hours. How far does it travel?",
        "answer": "Distance = Speed × Time\nDistance = 80 × 3 = 240 km. The train covers 240 km in 3 hours.",
        "amateur_feedback": "I like how the calculation is done step by step, but maybe add a note about what the distance means in real life.",
        "expert_feedback": "The answer correctly calculates the distance using the proper formula. Each step is logically presented and clear. To improve, connecting the formula to a real-world context would strengthen understanding.",
        "unified_feedback": "The solution correctly calculates the distance and explains each step clearly. To enhance clarity, include a brief note connecting the formula to real-world motion. Overall, this is a precise and well-structured response.",
        "self_critique": "The answer is accurate, but adding context would improve reader understanding.",
        "score": 0.90
    }
]

# Model Setup

## Expert Feedback Model

**Key Features**

1. **Answer Generation**: Produce an initial answer based on the given prompt.

2. **Feedback Evaluation**:

  * Scores answers using a predefined rubric (shown below) CLEAR papper, and explains how the answers can be improved

  * During feedback evaluation, the system follows a structured procedure:

      * Analyze the answer

      * Compose Feedback

      * Score and Output

  * For scoring rubric:

              
              • 1.0 — Excellent: insightful, emotionally engaging, and well-structured\n"
              •  0.7–0.9 — Good: mostly clear and thoughtful with minor issues\n"
              •  0.4–0.6 — Fair: has substance but lacks clarity or cohesion\n"
              •  0.1–0.3 — Weak: vague, underdeveloped, or emotionally flat\n"
              •  0.0 — Not helpful: irrelevant or incoherent\n"
              • -0.1 to -0.4 — Somewhat misleading or confusing\n"
              • -0.5 to -0.9 — Mostly incorrect, misrepresents the source\n"
              • -1.0 — Harmful or dangerously misleading\n\n"

3. **Feedback Merging**: Combines expert and amateur feedback, prioritizing expert insight

4. **Answer Revision**: improves answers based on feedback and self-critique

  * **Self-Critique**: Provides a brief reflection from the perspective of the answer's author, highlighting its strengths, identifying areas for improvement, and suggesting ways the answer could be enhanced in its own words.

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def truncate_to_first_paragraph_expert(text):
    for part in text.strip().split('\n\n'):
        paragraph = part.strip().split('\n')[0].strip()
        if paragraph:
            return paragraph
    return text.strip()

class ExpertFeedbackModel:
    def __init__(self, tokenizer, model, model_name):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = model
        self.tokenizer = tokenizer

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"Loaded {model_name} on {self.device}")

    def generate_text(self, prompt, max_new_tokens=300):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_answer(self, question, is_math = False, task_description=None, examples=None, max_new_tokens=500, max_words=300):
        question = question.strip()
        task_description = task_description.strip() if task_description else None

        one_shot_text = ""
        if is_math and examples and len(examples) > 0:
            one_shot_example = examples[0]
            one_shot_question = one_shot_example.get('question', '')
            one_shot_answer = one_shot_example.get('answer', '')
            one_shot_text = f"Question: {one_shot_question}\nAnswer: {one_shot_answer}\n\n"

        if is_math:
            task_text = f"Task Description: {task_description}\n" if task_description else ""

            prompt = (
                "You are a specialized math language model tasked with generating accurate, "
                "well-reasoned, step-by-step solutions to specific math problems.\n"
                "Your goal is to provide clear, precise, and fully self-contained explanations "
                "suitable for anyone seeking a solid understanding of the topic, regardless of prior background knowledge.\n"
                "When responding:\n"
                "  1) Identify the mathematical concept or principle involved.\n"
                "  2) Solve the problem step by step.\n"
                "  3) Present the solution in a logically structured and concise manner.\n"
                "Keep the tone professional, formal, concise and focused. Avoid filler words, emojis, or informal phrasing.\n"
                "Write your answer in a single paragraph.\n"
                f"Limit your response to approximately {max_words} words.\n\n"
                "Think carefully before responding. Format your output as follows:\n\n"
                "Answer: <one-paragraph response>\n\n"
                f"{task_text}"
                f"Question: {question}\n"
                "Answer:"
            )

        else:
            prompt = (
                  "You are an expert language model tasked with providing precise, well-reasoned, and informative responses.\n"
                  "Your goal is to deliver a clear, accessible, and accurate response in a formal yet approachable tone for a general audience with no assumed background knowledge.\n"
                  "The user is an informed individual seeking a reliable, well-reasoned, and self-contained explanation.\n"
                  "Your response will be used to inform readers seeking a foundational understanding of the topic.\n"
                  "When responding to the request: 1. Identify the core message. 2. Explain it in a way that is logically structured, accurate, and easy to understand.\n"
                  "Keep the tone professional and concise. Avoid jargon, filler words, and emojis.\n"
                  f"Limit your response to approximately {max_words} words. If a full explanation would be too long, summarize it while ensuring it remains self-contained and informative.\n"
                  "Write in a single paragraph with no lists, bullet points, or follow-up suggestions.\n"
                  "Think step by step to ensure your response improves clarity and aligns with the target audience.\n\n"
                  f"Task Description: {task_description}"
                  f"Question: {question}\n\n"
                  "Answer:"
              )


        response = self.generate_text(prompt, max_new_tokens)

        # print(f"Response: {response}")
        blocks = response.split("Answer:")

        if not is_math:
            if len(blocks) >= 2:
                answer = blocks[1].strip()
            else:
                answer = ""
        else:
            blocks = response.split("Answer:")
            if len(blocks) >= 3:
                answer = blocks[2].strip()
            else:
                answer = ""

        return truncate_to_first_paragraph_expert(answer)


    def generate_feedback(self, question, answer, is_math = False, examples = None):
        one_shot_text = ''
        if is_math:
            one_shot_example = examples[0]
            one_shot_question = one_shot_example.get('question', '')
            one_shot_answer = one_shot_example.get('answer', '')
            one_shot_score = one_shot_example.get('score', '')
            one_shot_feedback = one_shot_example.get('expert_feedback', '')
            one_shot_text = f"Question: {one_shot_question}\nAnswer: {one_shot_answer}\nScore: {one_shot_score}\nFeedback: {one_shot_feedback}\n"
        else:
            one_shot_text = (
                "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
                "Answer: The author uses the image of the broken mirror to represent how the protagonist feels fractured internally. "
                "Each mention of the mirror reflects a different stage of the character’s turmoil. "
                "Initially, the mirror is intact and clear, but as the story progresses, it becomes cracked and dusty, symbolizing the descent into depression and self-doubt. "
                "The mirror also highlights how the character perceives themselves — distorted and unclear due to emotional pain.\n\n"
                "Score: 0.85\n"
                "Feedback: This answer demonstrates a perceptive and coherent interpretation of the mirror as a symbol of the protagonist’s emotional state. "
                "The symbolic progression is clearly articulated, and the discussion of distortion is particularly effective. "
                "To improve, the analysis should incorporate specific textual evidence. "
                "Overall, this is a strong and emotionally insightful response.\n"
            )

        prompt = (
            "You are an expert feedback evaluator. Your task is to assess the quality and relevance of the answer provided below.\n"
            "The answer may involve subjective analysis (e.g., literature, interpretation) or objective reasoning (e.g., math, science).\n\n"
            "Follow these steps:\n"
            "1. Carefully analyze the answer. Internally identify its strengths, weaknesses, and areas for improvement.\n"
            "2. Write a clear, specific, and constructive paragraph of feedback that would help the user improve their response.\n"
            "3. Output your final **Score** and **Feedback** in the format shown below.\n\n"
            "Use a professional, concise, and objective tone. Do not include personal encouragements or vague generalities.\n"
            "Use the following scoring scale:\n"
            "  •  1.0 — Excellent: insightful, emotionally engaging, and well-structured\n"
            "  •  0.7–0.9 — Good: mostly clear and thoughtful with minor issues\n"
            "  •  0.4–0.6 — Fair: has substance but lacks clarity or cohesion\n"
            "  •  0.1–0.3 — Weak: vague, underdeveloped, or emotionally flat\n"
            "  •  0.0 — Not helpful: irrelevant or incoherent\n"
            "  • -0.1 to -0.4 — Somewhat misleading or confusing\n"
            "  • -0.5 to -0.9 — Mostly incorrect, misrepresents the source\n"
            "  • -1.0 — Harmful or dangerously misleading\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Score: <number>\n"
            "Feedback: <one-paragraph response>\n\n"
            f"{one_shot_text}"
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            "Score:"
        )

        response = self.generate_text(prompt, max_new_tokens=200).strip()
        # print(f"Response: {response}")
        score_blocks = response.split("Score:")
        if len(score_blocks) >= 3:
            score_text = score_blocks[3].strip().split()[0]
        else:
            score_text = None

        try:
            score = float(score_text)
        except (TypeError, ValueError):
            score = -1000

        feedback_blocks = response.split("Feedback:")
        feedback_blocks = [f for f in feedback_blocks if '<' not in f and '>' not in f]

        if len(feedback_blocks) >= 2:
            feedback_text_expert = truncate_to_first_paragraph_expert(feedback_blocks[1].strip())
        else:
            feedback_text_expert = ""

        # print(f"Feedback expert : {feedback_text_expert}")

        return feedback_text_expert, score


    def prompt_condense_feedback(self, expert_feedback, amateur_feedback):
        return (
            f"You are a feedback summarizer tasked with condensing two feedback comments on an answer.\n"
            f"Your goal is to generate a clear and concise summary by prioritizing the expert feedback while incorporating any useful or original points from the amateur feedback.\n"
            f"Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            f"The user is an informed individual seeking a balanced summary to improve their response.\n"
            f"Your summary should provide informative and actionable feedback.\n"
            f"In cases of disagreement, favor the expert’s insights and ensure the final version is coherent, constructive, and no more than 70 words.\n\n"
            f"You are given two feedback comments on an answer:\n"
            f"Your task is to write a unified feedback summary (max 70 words) that:\n"
            f"- Prioritizes insights from the expert feedback\n"
            f"- Incorporates any unique and constructive points from the amateur feedback\n"
            f"- Resolves disagreements by favoring the expert\n"
            f"- Uses contrastive reasoning to emphasize the strengths of the expert feedback\n\n"
            f"Think carefully before responding. Format your output as follows:\n\n"
            f"Final Merged Feedback: <one-paragraph response>\n\n"
            f"Expert Feedback: The argument is clear and well-structured, but the conclusion could be stronger with more supporting data. Consider adding one more example to illustrate the key point.\n"
            f"Amateur Feedback: I liked how the answer flows, but maybe shorten some sentences to make it punchier.\n"
            f"Final Merged Feedback: The answer is clear, with strong logical flow. To enhance impact, strengthen the conclusion by adding more supporting data and one illustrative example. Additionally, minor sentence shortening can improve readability without losing clarity.\n\n"
            f"Expert Feedback: {expert_feedback.strip()}\n"
            f"Amateur Feedback: {amateur_feedback.strip()}\n"
            f"Final Merged Feedback:"
        )



    def generate_unified_feedback(self, expert_feedback, amateur_feedback, max_new_tokens=200):
        if amateur_feedback == "Failed to generate meaningful feedback":
            return expert_feedback.strip()

        full_prompt = self.prompt_condense_feedback(expert_feedback, amateur_feedback)
        response = self.generate_text(full_prompt, max_new_tokens=max_new_tokens).strip()

        feedback_blocks = response.split("Final Merged Feedback:")

        feedback_blocks = [f.strip() for f in feedback_blocks if '<' not in f and '>' not in f and f.strip()]

        if len(feedback_blocks) >= 3:
            final_answer = truncate_to_first_paragraph_expert(feedback_blocks[2].strip())
        else:
            final_answer = truncate_to_first_paragraph_expert(response.strip())

        # print(f"Response: {response}")
        # print(f"Merged Feedback: {final_answer}")

        return final_answer



    def apply_feedback(self, question, answer, feedback, max_new_tokens=500):
        prompt = (
            "You are an expert language model tasked with producing precise, well-reasoned, and informative responses.\n"
            "Your goal is to revise the original answer using the provided feedback to create a response that is clear, accurate, and accessible.\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, logically structured, and self-contained explanation.\n"
            "Your revised response will help readers develop a foundational understanding of the topic.\n"
            "When crafting the response: 1. Identify the core message. 2. Present it in a logically structured, accurate, and easy-to-understand manner.\n"
            "Maintain a professional and concise tone. Avoid jargon, filler words, and emojis.\n"
            "Write your answer in a single paragraph.\n"
            "Limit your response to approximately 200 words.\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Revised Answer: <one-paragraph response>\n\n"
            "Revise the following answer using the feedback provided.\n"
            f"Question:{question}\n"
            f"Original Answer:{answer}\n"
            f"Feedback:{feedback}\n"
            "Revised Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens).strip()

        answer_blocks = response.split("Revised Answer:")

        answer_blocks = [a.strip() for a in answer_blocks if '<' not in a and '>' not in a and a.strip()]

        if len(answer_blocks) >= 2:
            final_answer = answer_blocks[1].strip()
        else:
            final_answer = response.strip()

        return truncate_to_first_paragraph_expert(final_answer)

    def apply_self_critique(self, question, answer, self_critique, max_new_tokens=500):
        prompt = (
            "You are an expert language model tasked with producing clear, accurate, and well-reasoned answers.\n"
            "Your goal is to revise the following answer based on the provided self-critique, "
            "ensuring the revision is logically sound, clear, and self-contained.\n\n"
            "Write in a formal yet approachable tone, suitable for a general audience with no assumed background knowledge.\n"
            "The user is an informed individual seeking a reliable, logically structured, and self-contained explanation.\n\n"
            "When crafting the response:\n"
            "1. Incorporate all insights from the self-critique.\n"
            "2. Improve clarity and logical flow.\n"
            "3. Correct any errors or ambiguities in the original answer.\n"
            "4. Keep the answer concise but thorough.\n\n"
            "Limit your response to approximately 200 words.\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Final Answer: <one-paragraph response>\n\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            f"Self-Critique: {self_critique}\n"
            "Final Answer:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens).strip()

        answer_blocks = response.split("Final Answer:")

        answer_blocks = [a.strip() for a in answer_blocks if '<' not in a and '>' not in a and a.strip()]

        if len(answer_blocks) >= 2:
            final_answer = answer_blocks[1].strip()
        else:
            final_answer = ""

        return truncate_to_first_paragraph_expert(final_answer)


    def generate_self_critique(self, question, answer, max_new_tokens=150):
        prompt = (
            "You are the author of the provided answer. Your task is to critically evaluate your own response in a concise, paragraph-style self-critique.\n"
            "You should analyze your answer for clarity, accuracy, completeness, and relevance to the question.\n"
            "You should identify the strengths of your response and explain why certain parts effectively address the question.\n"
            "You should also highlight weaknesses or gaps, and suggest how you could have improved the answer.\n"
            "You should not rewrite the answer; focus on evaluating it objectively from your own perspective.\n"
            "You should write in a formal yet approachable tone, suitable for a general audience.\n"
            "You should limit your critique to approximately 70 words, focusing on the most important points.\n\n"
            "Format the output as follows:\n"
            "Self-Critique: <one-paragraph response>\n\n"
            "Question: What are the main factors contributing to climate change?\n"
            "Answer: Climate change is caused primarily by human activities, such as burning fossil fuels and deforestation, which increase greenhouse gas concentrations in the atmosphere. Natural factors like volcanic eruptions and solar variability also play minor roles.\n"
            "Self-Critique: I correctly identified human activities as the primary driver of climate change, which is accurate and relevant. I acknowledged natural factors, offering balance. However, I could improve by naming specific greenhouse gases and their impacts, which would make the response more complete. Overall, my answer is accurate but could be more detailed and precise.\n"
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            "Self-Critique:"
        )

        response = self.generate_text(prompt, max_new_tokens=max_new_tokens).strip()

        critique_blocks = response.split("Self-Critique:")

        critique_blocks = [c.strip() for c in critique_blocks if '<' not in c and '>' not in c and c.strip()]

        if len(critique_blocks) >= 3:
            final_answer = truncate_to_first_paragraph_expert(critique_blocks[2].strip())
        else:
            final_answer = ""

        # print(f"Self-Critique: {final_answer}")

        return final_answer



    def improve_answer_with_feedback_and_critique(self, question, answer, epochs, iterations, target_score=1.0, mode="train", is_math = False, examples = None):
        KD = True if mode == "train" else False
        original_feedback_dict = self.network.generate_combined_feedback(question, answer, is_math, examples, epochs, iterations, 0.65, KD)

        expert_feedback = original_feedback_dict.get("expert_feedback", "")
        amateur_feedback = original_feedback_dict.get("amateur_feedback", "")
        initial_score = float(original_feedback_dict.get("combined_score", -1.0))
        print(f"Expert Feedback: {expert_feedback}")
        print(f"Amateur Feedback: {amateur_feedback}")

        if initial_score >= target_score:
            return answer.strip(), expert_feedback, amateur_feedback, ""

        condensed_feedback = self.generate_unified_feedback(expert_feedback, amateur_feedback)
        feedback_paragraph = condensed_feedback.strip().split('\n\n')[0].strip()
        print(f"Condensed: {feedback_paragraph}")

        improved = self.apply_feedback(question, answer, feedback_paragraph)
        print(f"Improved: {improved}")

        critique = self.generate_self_critique(question, improved)

        refined = self.apply_self_critique(question, improved, critique)
        print(f"Refined: {refined}")

        return refined.strip(), expert_feedback, amateur_feedback, feedback_paragraph

In [ ]:
expert_model = ExpertFeedbackModel(teacher_tokenizer, teacher_model, teacher_model_name)

## Amateur Feedback Model

**Key Features**

* **Feedback Generation**:

  * Produces detailed feedback on answers based on criteria such as relevance to the prompt, clarity, and completeness.

  * For the feedback generation, the model uses NLP techniques to produce feedback while incorporating a scoring head to evaluate answer quality



In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn as nn

def truncate_to_first_paragraph_amateur(text):
    for part in text.strip().split('\n\n'):
        paragraph = part.strip().split('\n')[0].strip()
        if paragraph:
            return paragraph
    return text.strip()


class AmateurFeedbackModel:
    def __init__(self, tokenizer, model, model_name, use_bf16 = True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = tokenizer

        self.model = model
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        hidden_dim = self.model.config.hidden_size
        self.model_dtype = torch.bfloat16 if use_bf16 else torch.float32
        self.score_head = nn.Linear(hidden_dim, 1, dtype=self.model_dtype).to(self.device)
        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

        self.student_frozen = False

    def generate_text(self, prompt, max_new_tokens=300):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        model = self.model.module if isinstance(self.model, torch.nn.DataParallel) else self.model

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_feedback(self, question, answer, is_math = False, examples=None):
        one_shot_text = ''
        if is_math:
            one_shot_example = examples[0]
            one_shot_question = one_shot_example.get('question', '')
            one_shot_answer = one_shot_example.get('answer', '')
            one_shot_feedback = one_shot_example.get('expert_feedback', '')
            one_shot_text = f"Question: {one_shot_question}\nAnswer: {one_shot_answer}\nFeedback: {one_shot_feedback}\n"
        else:
            one_shot_text = (
                "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
                "Answer: The author uses the image of the broken mirror to represent how the protagonist feels fractured internally. "
                "Each mention of the mirror reflects a different stage of the character’s turmoil. "
                "Initially, the mirror is intact and clear, but as the story progresses, it becomes cracked and dusty, symbolizing the descent into depression and self-doubt. "
                "The mirror also highlights how the character perceives themselves — distorted and unclear due to emotional pain.\n\n"
                "Feedback: This answer demonstrates a perceptive and coherent interpretation of the mirror as a symbol of the protagonist’s emotional state. "
                "The symbolic progression is clearly articulated, and the discussion of distortion is particularly effective. "
                "To improve, the analysis should incorporate specific textual evidence. "
                "Overall, this is a strong and emotionally insightful response.\n\n"
                )
        prompt = (
            "You are an expert feedback evaluator. Your task is to assess the quality and relevance of the provided answer.\n"
            "The answer may involve subjective analysis (e.g., literature, interpretation) or objective reasoning (e.g., math, science).\n\n"
            "Instructions:\n"
            "1. Identify the answer’s strengths, weaknesses, and areas for improvement.\n"
            "2. Write a clear, specific, and constructive paragraph of feedback.\n"
            "3. Provide the answer in plain text only, with no special characters or markdown.\n"
            "4. Use a professional, concise, and objective tone. Avoid encouragements or vague comments.\n"
            "5. Keep the response around 70 words.\n\n"
            "Think carefully before responding. Format your output as follows:\n\n"
            "Feedback: <one-paragraph response>\n\n"
            f"{one_shot_text}"
            f"Question: {question}\n"
            f"Answer: {answer}\n\n"
            "Feedback:"
        )

        response = self.generate_text(prompt, max_new_tokens=200).strip()
        # print(f"Amateur Feedback Response: {response}")
        feedback_blocks = response.split("Feedback:")
        feedback_blocks = [f for f in feedback_blocks if '<' not in f and '>' not in f]

        if len(feedback_blocks) >= 3:
            feedback_text = truncate_to_first_paragraph_amateur(feedback_blocks[2].strip())
        else:
            feedback_text = ""

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model(**inputs, output_hidden_states=True, return_dict=True)
        last_hidden_state = outputs.hidden_states[-1][:, -1, :]
        last_hidden_state = last_hidden_state.to(self.score_head.weight.dtype)
        raw_score = self.score_head(last_hidden_state)
        score_logits = torch.tanh(raw_score)
        score_value = score_logits.detach().item()

        return feedback_text, score_value, score_logits

    def prepare_inputs_and_labels(self, prompt, answer, expert_feedback, is_math = False, examples = None):
        one_shot_text = ''
        if is_math:
            one_shot_example = examples[0]
            one_shot_question = one_shot_example.get('question', '')
            one_shot_answer = one_shot_example.get('answer', '')
            one_shot_feedback = one_shot_example.get('expert_feedback', '')
            one_shot_text = f"Question: {one_shot_question}\nAnswer: {one_shot_answer}\nFeedback: {one_shot_feedback}\n"
        else:
            one_shot_text = (
                "Question: How does the author use symbolism in the story to convey the protagonist's emotional journey?\n"
                "Answer: The author uses the image of the broken mirror to represent how the protagonist feels fractured internally. "
                "Each mention of the mirror reflects a different stage of the character’s turmoil. "
                "Initially, the mirror is intact and clear, but as the story progresses, it becomes cracked and dusty, symbolizing the descent into depression and self-doubt. "
                "The mirror also highlights how the character perceives themselves — distorted and unclear due to emotional pain.\n\n"
                "Feedback: This answer demonstrates a perceptive and coherent interpretation of the mirror as a symbol of the protagonist’s emotional state. "
                "The symbolic progression is clearly articulated, and the discussion of distortion is particularly effective. "
                "To improve, the analysis should incorporate specific textual evidence. "
                "Overall, this is a strong and emotionally insightful response.\n\n"
                )

        input_text = (
            "You are an expert feedback evaluator. Your task is to assess the quality and relevance of the provided answer.\n"
            "The answer may involve subjective analysis (e.g., literature, interpretation) or objective reasoning (e.g., math, science).\n\n"
            "Instructions:\n"
            "1. Identify the answer’s strengths, weaknesses, and areas for improvement.\n"
            "2. Write a clear, specific, and constructive paragraph of feedback.\n"
            "3. Use this format: Feedback: <one-paragraph response>.\n"
            "4. Provide the answer in plain text only, with no special characters or markdown.\n"
            "5. Use a professional, concise, and objective tone. Avoid encouragements or vague comments.\n"
            "6. Keep the response around 70 words.\n\n"
            f"{one_shot_text}"
            f"Question: {prompt}\n"
            f"Answer: {answer}\n\n"
            "Feedback:"
        )

        tokenized_input = self.tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024)
        input_ids = tokenized_input.input_ids[0].to(self.device)

        tokenized_feedback = self.tokenizer(expert_feedback, return_tensors="pt", truncation=True, max_length=1024)
        feedback_ids = tokenized_feedback.input_ids[0].to(self.device)

        input_ids = torch.cat([
            input_ids,
            torch.full((len(feedback_ids),), self.tokenizer.pad_token_id, device=self.device)
        ])

        labels = torch.full_like(input_ids, -100)
        labels[-len(feedback_ids):] = feedback_ids

        attention_mask = torch.cat([
            tokenized_input.attention_mask[0].to(self.device),
            torch.ones(len(feedback_ids), device=self.device)
        ])

        # print("Input IDs length:", len(input_ids))
        # print("Feedback IDs length:", len(feedback_ids))
        # print("Masked part (labels=-100):", self.tokenizer.decode(input_ids[:-len(feedback_ids)], skip_special_tokens=True))
        # print("Feedback part (visible in labels):", self.tokenizer.decode(labels[-len(feedback_ids):], skip_special_tokens=True))

        return (
            input_ids.unsqueeze(0).to(self.device),
            labels.unsqueeze(0).to(self.device),
            attention_mask.unsqueeze(0).to(self.device)
        )

    def freeze_student_model(self):
        for param in self.model.parameters():
            param.requires_grad = False
        self.student_frozen = True

In [ ]:
amateur_feedback_model = AmateurFeedbackModel(student_tokenizer, student_model, student_model_name)

**A debugging JSON file for feedback generation and testing the extraction of input IDs and labels.** -> [amateur_results](https://drive.google.com/file/d/1FQT2OFiLFEklNm_GqcQeBG6vbzZtYh2Q/view?usp=sharing)

## Comparative Testing Framework

The `RefinedAnswerEvaluator` and utility functions provide a framework for comparative testing of *original* vs *refined* answers using multiple evaluation criteria.

  * **LLM judgement**

  * **Readability**

  * **Engagement**

  * **Semantic similarity**

  * **Fluency**

  * **Consistency**

For each answer pair, we aggregate the metric scores into a feature vecto representing the evaluation based on these criteria. The LLM's judgement is treated as a separate candidate. We then compute an `if_matched`, which is `1` when the LLM judgemnet aligns with the metric judgement, and `0` otherwise.

Next, we perform a p-test on the `if_matched` values to assess whether the refined answers are significantly better than the original ones. If the resulting p-value < 0.05, we conclude that the refined answers are statistically significantly better.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
import torch
import re

class RefinedAnswerEvaluator:
    def __init__(self, tokenizer, model, use_bf16=True):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = tokenizer
        self.model_dtype = torch.bfloat16 if use_bf16 else torch.float16 if self.device == "cuda" else torch.float32
        self.model = model
        print(f"Loaded {model_name} on {self.device} with dtype {self.model_dtype}")

    def generate_text(self, prompt, max_new_tokens=50, temperature=0.3):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=temperature,
            top_p=0.5,
            pad_token_id=self.tokenizer.eos_token_id
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)


    def is_refined_better(self, original_answer, refined_answer):
        prompt = (
            "You are a teacher grading answers to evaluate which sentence is better.\n"
            "Your goal is to provide a 'Yes' or 'No' answer.\n"
            "Criteria to evaluate the refined answer over the original answer:\n"
            "- Clarity: Is the refined sentence easier to understand?\n"
            "- Persuasiveness: Does the refined answer convince the reader more?\n"
            "- Engagement: Does the refined answer keep the reader interested?\n\n"
            "Original answer: The cat sat on the mat.\n"
            "Refined answer: The cat comfortably rested on the mat.\n"
            "Better? Yes\n\n"
            "Original answer: Water boils at 100 degrees Celsius.\n"
            "Refined answer: Water boils at 0 degrees Celsius.\n"
            "Better? No\n\n"
            f"Original answer: {original_answer}\n"
            f"Refined answer: {refined_answer}\n"
            "Better?"
        )

        output = self.generate_text(prompt, max_new_tokens=80)
        # print(f"Output: {output}")

        matches = re.findall(r"\bYes\b|\bNo\b", output, re.IGNORECASE)
        # print(f"All matches found: {matches}")

        if matches:
            return matches[4].lower() == "yes"
        return False

    def generate_engagement_score(self, text, max_new_tokens=10):

        prompt = (
            "You are a teacher grading answers to evaluate how engaging a sentence or paragraph is.\n"
            "On a scale of 1 to 10, use the following criteria:\n"
            "1: Extremely boring; reader would likely lose interest immediately.\n"
            "2: Very unengaging; dull and lacks clarity or appeal.\n"
            "3: Somewhat unengaging; few interesting elements.\n"
            "4: Slightly engaging; may hold attention briefly.\n"
            "5: Moderately engaging; neither particularly boring nor exciting.\n"
            "6: Fairly engaging; keeps the reader’s attention most of the time.\n"
            "7: Engaging; reader finds it interesting and clear.\n"
            "8: Very engaging; strong clarity, persuasiveness, and appeal.\n"
            "9: Extremely engaging; highly compelling and keeps reader fully interested.\n"
            "10: Exceptionally engaging; captivating, persuasive, and very clear, leaving a strong impact.\n\n"

            "Text: 'The cat sat quietly on the mat.'\n"
            "Score: 3\n\n"

            f"Text: {text}.\n"
            "Score:"
        )

        output = self.generate_text(prompt, max_new_tokens)
        # print(f"Output: {output}")

        score_matches = re.findall(r"Score\s*:\s*(\d+\.?\d*)", output)
        if score_matches:
            return float(score_matches[-1])
        return 0

In [ ]:
evaluator = RefinedAnswerEvaluator(teacher_tokenizer, teacher_model)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_1samp
from collections import Counter
import math
import spacy
from textstat import flesch_reading_ease
from sentence_transformers import SentenceTransformer, util

nlp = spacy.load("en_core_web_sm")
emb_model = SentenceTransformer('all-MiniLM-L6-v2')

def normalize_readability(score, text, informativeness, alpha = 0.5):
    base = textstat.flesch_reading_ease(text)

    adjusted = (1 - alpha) * base + alpha * informativeness
    return adjusted

def engagement_score(text, evaluator):
    doc = nlp(text)
    verbs = sum(1 for token in doc if token.pos_ == "VERB")
    adjectives = sum(1 for token in doc if token.pos_ == "ADJ")
    length = len(doc)

    spacy_score = (verbs * 1.5 + adjectives * 1.0) / max(length, 1)

    llm_score_raw = evaluator.generate_engagement_score(text)
    llm_score = (llm_score_raw - 1) / 9

    combined_score = 0.5 * spacy_score + 0.5 * llm_score
    return combined_score

def specificity_score(text):
    doc = nlp(text)
    content_words = sum(1 for token in doc if token.pos_ in {"NOUN", "VERB", "ADJ", "ADV"})
    return content_words / max(len(doc), 1)

def informativeness_score(text):
    tokens = [t.text.lower() for t in nlp(text) if not t.is_punct and not t.is_space]
    counts = Counter(tokens)
    total = sum(counts.values())
    if total == 0:
        return 0
    probs = [c/total for c in counts.values()]
    entropy = -sum(p * math.log(p, 2) for p in probs)
    return entropy / math.log(len(counts)+1, 2)

def compute_semantic_similarity(original_answer, refined_answer):
    emb_orig = emb_model.encode(original_answer, convert_to_tensor=True)
    emb_refined = emb_model.encode(refined_answer, convert_to_tensor=True)
    similarity = util.cos_sim(emb_orig, emb_refined).item()
    return similarity


def compute_perplexity(text, teacher_model, teacher_tokenizer):
    tokens = teacher_tokenizer(text, return_tensors="pt")
    input_ids = tokens["input_ids"]

    with torch.no_grad():
        outputs = teacher_model(input_ids, labels=input_ids)
        loss = outputs.loss

    return torch.exp(loss).item()

def is_refined_more_fluent(original_answer, refined_answer, model, tokenizer):
    original_ppl = compute_perplexity(original_answer, model, tokenizer)
    refined_ppl = compute_perplexity(refined_answer, model, tokenizer)

    return refined_ppl < original_ppl

def is_consistent(premise, hypothesis, threshold=0.5):
    inputs = nli_tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = nli_model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        contradiction_prob = probs[0].item()
        entailment_prob = probs[2].item()

        is_consistent_flag = entailment_prob >= threshold
    return is_consistent_flag

def comparative_testing_results_log(pairs, evaluator):
    results = []
    for original, refined in pairs:
        engagement_orig = engagement_score(original, evaluator)
        engagement_refined = engagement_score(refined, evaluator)

        specificity_orig = specificity_score(original)
        specificity_refined = specificity_score(refined)

        informativeness_orig = informativeness_score(original)
        informativeness_refined = informativeness_score(refined)

        readability_orig = normalize_readability(
        flesch_reading_ease(original), original, informativeness_orig)

        readability_refined = normalize_readability(
        flesch_reading_ease(refined), refined, informativeness_refined)

        semantic_similarity = compute_semantic_similarity(original, refined)

        llm_pref = 1 if evaluator.is_refined_better(original, refined) else -1

        is_refined_easier_to_read = 1 if is_refined_more_fluent(original, refined, teacher_model, teacher_tokenizer) else 0
        is_refined_consistent = 1 if is_consistent(refined, original) else 0

        results.append({
            "original_readability": readability_orig,
            "refined_readability": readability_refined,
            "original_engagement": engagement_orig,
            "refined_engagement": engagement_refined,
            "original_specificity": specificity_orig,
            "refined_specificity": specificity_refined,
            "specificity_better": int(specificity_refined > specificity_orig),
            "original_informativeness": informativeness_orig,
            "refined_informativeness": informativeness_refined,
            "informativeness_better": int(informativeness_refined > informativeness_orig),
            "semantic_similarity": semantic_similarity,
            "llm_pref": llm_pref,
            "is_refined_fluent": is_refined_easier_to_read,
            "is_consistent": is_refined_consistent,
            "readability_better": int(readability_refined > readability_orig),
            "engagement_better": int(engagement_refined > engagement_orig),
            "fluent_better": is_refined_easier_to_read,
        })

    return results


def evaluate_pairs_with_frequentist(log_results, output_file="statistical_testing.json", alpha=0.05):
    votes = []
    decisions = []

    features = [
        "readability_better",
        "engagement_better",
        "specificity_better",
        "informativeness_better",
        "fluent_better"
    ]

    metric_data = []

    for r in log_results:
        metric_votes = [r.get(f, 0) for f in features]

        vote_sum = sum(metric_votes)
        metric_majority_vote = 1 if vote_sum > 0 else -1

        llm_pref = r.get("llm_pref", 0)

        if_match = int(llm_pref == metric_majority_vote)
        votes.append(if_match)

        metric_data.append(metric_votes + [llm_pref])

        decisions.append({
            "llm_pref": llm_pref,
            "metric_majority_vote": metric_majority_vote,
            "match": bool(if_match)
        })

    t_stat, p_value = ttest_1samp(votes, 0.5, alternative='greater')
    final_decision = bool(p_value < alpha)

    output_data = {
        "votes": votes,
        "decisions": decisions,
        "p_value": float(p_value),
        "final_decision": final_decision
    }

    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=2)

    df_for_corr = pd.DataFrame(metric_data, columns=features + ["llm_pref"])

    corr_matrix = df_for_corr.corr()

    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.show()

    return final_decision

**IMG of heatmap** --> [heat_map](https://drive.google.com/file/d/1WlI_k7Xmjnlbf_tmRFNqaXNNX9dAW-EJ/view?usp=sharing)

## Knowledge Distillation: Loss Functions & Logging Utilities

This module provides the core loss computations and supporting utilities for KD. It covers both the definition of loss functions used during trainign and thte tools needed for analysis, logging, and visualization.

* **Loss Functions**

  * `compute_lm_loss` : standard language modeling loss

  * `compute_hidden_loss`: aligns hidden featires between student and teacher

  * `compute_scoring_loss` : matches score predictions across models

  * `compute_logit_standardization` : logit based KD with normalization (as described in[Logit Standardization in Knowledge Distillatiom](https://arxiv.org/pdf/2403.01427))

  * `compute_rkd_loss` : relational KD loss for structural representation learning (as described in [distance-wise distillation loss](https://arxiv.org/pdf/1904.05068))

  * `compute_all_losses_vectorized` : collects all loss values into a single vector for efficiency

  * `compute_avg_loss` : aggregates per-epoch loss values

* **Analysis & Visualization**

  * `visualize_loss_functions` - plots per epoch training loss trends

  * `compute_bert_score`

  * `unpack_feedback_and_scores`

* **Logging & Conversion**

  * `save_feedback_to_csv`

  * `convert_csv_to_json`

In [ ]:
import random
random.seed(0)
distill_batch = random.sample([s['feedback'] for s in expert_datasets], 4)

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
import os
import csv

def compute_lm_loss(teacher_feedback, student, prompt, answer, device, model_dtype, is_math = False, examples = None):
    lm_input_ids, lm_labels, lm_attention_mask = student.prepare_inputs_and_labels(
                            prompt, answer, teacher_feedback, is_math, examples)

    lm_input_ids = lm_input_ids.to(device)
    lm_labels = lm_labels.to(device)
    lm_attention_mask = lm_attention_mask.to(device)

    lm_student_out = student.model(
        input_ids=lm_input_ids,
        attention_mask=lm_attention_mask,
        labels=lm_labels,
        output_hidden_states=True,
        return_dict=True
    )

    lm_loss = lm_student_out.loss.to(model_dtype)
    return lm_loss


def compute_hidden_loss(ref_text, student_tokenizer, teacher_tokenizer, student, teacher, align_hidden, device, model_dtype):
    student_inputs = student_tokenizer(ref_text, return_tensors="pt", truncation=True, padding=True).to(device)
    teacher_inputs = teacher_tokenizer(ref_text, return_tensors="pt", truncation=True, padding=True).to(device)

    student_out = student.model(**student_inputs, output_hidden_states=True, return_dict=True)
    with torch.no_grad():
        teacher_out = teacher.model(**teacher_inputs, output_hidden_states=True, return_dict=True)

    student_hidden, teacher_hidden = student_out.hidden_states, teacher_out.hidden_states
    num_student_layers, num_teacher_layers = len(student_hidden), len(teacher_hidden)
    layer_ids = torch.linspace(0, num_teacher_layers-1, num_student_layers).long()

    hidden_loss = 0.0

    for i, t_idx in enumerate(layer_ids):
        s_mean = student_hidden[i].to(model_dtype).mean(dim=1)
        t_mean = teacher_hidden[t_idx].to(model_dtype).mean(dim=1)

        s_norm = s_mean / s_mean.norm(dim=1, keepdim=True)
        t_norm = t_mean / t_mean.norm(dim=1, keepdim=True)

        s_aligned = align_hidden(s_norm)

        hidden_loss += F.mse_loss(s_aligned, t_norm, reduction='sum')

    hidden_loss = hidden_loss / num_student_layers

    return hidden_loss

def compute_scoring_loss(score_pred, teacher_score, model_dtype, device):
    # score_pred = student.score_head(student_hidden.to(device).to(model_dtype))
    teacher_score_tensor = torch.tensor(
        [teacher_score] * score_pred.size(0),
        dtype=model_dtype,
        device=device
    )

    scoring_loss = F.mse_loss(score_pred.squeeze(-1), teacher_score_tensor)
    return scoring_loss

def z_score(x, tau=1.0, eps=1e-8):
    mean = x.mean(dim=1, keepdim=True)
    std = x.std(dim=1, keepdim=True) + eps
    return (x - mean) / std / tau

# tau=1.0 -> baseline
def compute_logit_standardization(student, teacher, student_tokenizer, teacher_tokenizer, projection_layer, ref_text, tau=4.0):
    student_inputs = student_tokenizer(ref_text, return_tensors="pt", truncation=True).to(student.device)
    student_outputs = student.model(**student_inputs)
    student_logits = student_outputs.logits.to(torch.float32)
    student_logits_projected = projection_layer(student_logits)

    with torch.no_grad():
        teacher_inputs = teacher_tokenizer(ref_text, return_tensors="pt", truncation=True).to(student.device)
        teacher_outputs = teacher.model(**teacher_inputs)
        teacher_logits = teacher_outputs.logits.to(torch.float32)

    min_len = min(student_logits_projected.size(1), teacher_logits.size(1))
    student_logits_projected = student_logits_projected[:, :min_len, :]
    teacher_logits = teacher_logits[:, :min_len, :]

    teacher_logits_std = z_score(teacher_logits, tau)
    student_logits_std = z_score(student_logits_projected, tau)

    teacher_distribution = F.softmax(teacher_logits_std, dim=-1)
    student_distribution = F.log_softmax(student_logits_std, dim=-1)

    logits_loss = F.kl_div(student_distribution, teacher_distribution, reduction='batchmean') * (tau ** 2)

    return logits_loss

def pick_proportional_layers(student_total, teacher_total, n_layers=3):
    student_layers = [
        round(i * student_total / n_layers) for i in range(1, n_layers + 1)
    ]

    teacher_layers = [
        round(l * teacher_total / student_total) for l in student_layers
    ]
    return student_layers, teacher_layers

def compute_all_losses_vectorized(
    loss_config,
    prompt,
    answer,
    student,
    teacher,
    student_tokenizer,
    teacher_tokenizer,
    align_hidden_for_hidden_loss,
    projection_layer,
    device=None,
    is_math = False,
    examples=None,
    model_dtype=torch.float32
):
      device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

      student_feedback, student_score_value, student_score_logit = student.generate_feedback(
          prompt, answer, is_math, examples
      )
      with torch.no_grad():
          teacher_feedback, teacher_score_value = teacher.generate_feedback(
              prompt, answer, is_math=is_math, examples=examples
          )

      active_losses, _ = loss_config.get_active_losses()

      losses_by_name = {}

      if "lm_loss" in active_losses:
          lm_loss = compute_lm_loss(
              teacher_feedback, student, prompt, answer, device, model_dtype, is_math, examples
          )
          losses_by_name["lm_loss"] = lm_loss

      if "hidden_loss" in active_losses:
          hidden_loss = compute_hidden_loss(
              teacher_feedback, student_tokenizer, teacher_tokenizer,
              student, teacher, align_hidden_for_hidden_loss, device, model_dtype
          )
          losses_by_name["hidden_loss"] = hidden_loss

      if "scoring_loss" in active_losses:
          teacher_score_t = torch.as_tensor(teacher_score_value, device=device, dtype=model_dtype)
          scoring_loss = compute_scoring_loss(
              student_score_logit, teacher_score_t, model_dtype, device
          )
          losses_by_name["scoring_loss"] = scoring_loss

      if "logit_loss" in active_losses:
          logit_loss = compute_logit_standardization(
              student, teacher, student_tokenizer, teacher_tokenizer,
              projection_layer, teacher_feedback, tau=4.0
          )
          losses_by_name["logit_loss"] = logit_loss

      student_loss_vector = torch.stack(
          [losses_by_name[name].to(model_dtype) for name in active_losses]
      ).to(device)

      return {
          "loss_vector": student_loss_vector,
          "student_feedback": student_feedback,
          "teacher_feedback": teacher_feedback,
          "student_score": student_score_value,
          "teacher_score": teacher_score_value
      }


def unpack_feedback_and_scores(result):
    return (
        result["student_feedback"],
        result["teacher_feedback"],
        result["student_score"],
        result["teacher_score"]
    )


def safe_to_numpy(data):
    def to_float(x):
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().item() if x.numel() == 1 else x.detach().cpu().tolist()
        elif isinstance(x, list):
            return [to_float(y) for y in x]
        else:
            return float(x)
    return np.array(to_float(data))

def visualize_student_performance(
    student_performance,
    stop_threshold,
    loss_names=None,
    recent_window=None,
    robust=True,
    q_low=5, q_high=95,
    min_range=5.0
):
    student_performance = safe_to_numpy(student_performance)
    student_performance = np.array(student_performance, dtype=float)

    num_batches, num_losses = student_performance.shape
    x = np.arange(num_batches)

    if loss_names is None:
        loss_names = [f"Loss {i+1}" for i in range(num_losses)]

    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(num_losses, 1, figsize=(9, 10), sharex=True)
    if num_losses == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        gains = student_performance[:, i]
        color = colors[i % len(colors)]

        view = gains[-recent_window:] if (recent_window is not None and recent_window > 0) else gains

        if robust and len(view) > 1:
            lo = np.nanpercentile(view, q_low)
            hi = np.nanpercentile(view, q_high)
        else:
            lo = np.nanmin(view)
            hi = np.nanmax(view)

        if not np.isfinite(lo): lo = 0.0
        if not np.isfinite(hi): hi = 0.0
        if hi - lo < min_range:
            mid = 0.5 * (hi + lo)
            lo, hi = mid - 0.5 * min_range, mid + 0.5 * min_range

        pad = 0.1 * (hi - lo)
        y_min, y_max = lo - pad, hi + pad

        ax.plot(x, gains, marker='o', linestyle='-', alpha=0.25, color=color)
        ax.plot(x, gains, marker='o', linestyle='-', color=color, label=f'{loss_names[i]}', zorder=3)

        if y_min <= stop_threshold <= y_max:
            ax.axhline(stop_threshold, color='red', linestyle='--', label='Stop Threshold', zorder=2)
        else:
            ax.text(0.99, 0.02, "Stop thr. off-scale", transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=8, color='red')

        ax.set_ylim(y_min, y_max)
        ax.set_ylabel('Relative Gain (%)')
        ax.set_title(f'{loss_names[i]} Over Time')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left')

    axes[-1].set_xlabel('Batch / Epoch')
    fig.tight_layout()
    plt.show()

# Threshold Policy

In [ ]:
from dataclasses import dataclass
from collections import deque
from tabulate import tabulate
import torch
import hashlib
import numpy as np

def recalibrate_baselines(baseline_losses, current_losses, minimum_relative_gain, verbose=True):
    def _to_float(x):
        if isinstance(x, torch.Tensor):
            if x.numel() == 1:
                return float(x.detach().cpu().item())
            x = x.detach().cpu().numpy()
            return float(np.asarray(x).mean())
        return float(x)

    baseline_losses = [_to_float(v) for v in baseline_losses]
    current_losses  = [_to_float(v) for v in current_losses]

    new_baselines = []
    for i, (pb, cl) in enumerate(zip(baseline_losses, current_losses)):
        min_allowed = cl * (1 - minimum_relative_gain / 100.0)

        if pb > cl and pb <= (1 + minimum_relative_gain / 100.0) * cl:
            status, new_val = "KEEP (pb > cl)", pb
        elif pb >= min_allowed and pb < cl:
            status, new_val = "KEEP (pb between min_allowed and cl)", pb
        else:
            status = f"UPDATE (pb={pb:.4f}, cl={cl:.4f} -> min_allowed={min_allowed:.4f})"
            new_val = min_allowed

        new_baselines.append(new_val)

        if verbose:
            print(
              f" Loss {i}: PrevBaseline={pb:.4f}, CurrLoss={cl:.4f}, "
              f"MinAllowed={min_allowed:.4f} -> NewBaseline={new_val:.4f} [{status}]"
            )
    return new_baselines


@dataclass
class ThresholdConfig:
    strategy = "relative"
    ema_alpha = 0.1
    minimum_relative_gain = 10.0
    relative_gain_stop_threshold= 90.0

class AdaptiveLossWeighting:
    def __init__(self, num_losses, beta=1.0, device="cuda"):
        self.num_losses = num_losses
        self.beta = beta
        self.previous_losses = None
        self.device = device

    def compute_weights(self, current_losses, in_KD = False):
        current_losses_tensor = torch.tensor(current_losses, device=self.device, dtype=torch.float32)
        if self.previous_losses is None:
            weights = torch.ones_like(current_losses_tensor) / self.num_losses
        else:
            loss_deltas = current_losses_tensor - self.previous_losses
            weights = torch.softmax(self.beta * loss_deltas, dim=0)

        if not in_KD:
            self.previous_losses = current_losses_tensor.detach()
        return weights

class BatchLossTracker:
    def __init__(self, initial_losses, config):
        self.baseline_losses = None
        self.past_tracking_losses_per_batch = []
        self.config = config
        self.og_losses = initial_losses

    def update_baselines(self, current_losses, verbose=True):
        if self.baseline_losses is None:
            self.baseline_losses = current_losses
            self.past_tracking_losses_per_batch.append(current_losses)
            return self.baseline_losses

        new_baselines = recalibrate_baselines(self.baseline_losses, current_losses, self.config.minimum_relative_gain)

        self.baseline_losses = new_baselines
        self.past_tracking_losses_per_batch.append(current_losses[:])

        if verbose:
            print(f" Final baselines: {self.baseline_losses}\n")

        return self.baseline_losses


class AdaptiveWeightedKDPolicyEMA:
    def __init__(self, num_losses, initial_losses=None, beta=1.0, device="cuda", window_size=10):
        self.config = ThresholdConfig()
        self.adaptive_weighting = AdaptiveLossWeighting(num_losses, beta, device)
        self.device = device
        self.num_losses = num_losses
        self.window_size = window_size
        self.baseline_history = deque(maxlen=window_size)
        self.weights = []
        self.batch_loss_tracker_map = {}
        self.student_performance = []
        self.ladder_steps = list(range(10, 81, 10))
        self.current_ladder_idx = 0

        if initial_losses is not None:
            self.baseline_history.append(initial_losses[:])
            self.baseline_losses = initial_losses[:]
            self.ema_thresholds = [self.config.minimum_relative_gain] * len(initial_losses)
            self.weights = self.adaptive_weighting.compute_weights(self.baseline_losses).cpu().tolist()
        else:
            self.baseline_losses = None
            self.ema_thresholds = None

    def _to_float_list(self, tensor_or_list):
        if isinstance(tensor_or_list, torch.Tensor):
            tensor_or_list = tensor_or_list.detach().cpu().tolist()
        return [float(v) for v in tensor_or_list]

    def _update_ema_thresholds(self, relative_gains):
        if self.ema_thresholds is None:
            self.ema_thresholds = relative_gains[:]
        else:
            alpha = self.config.ema_alpha
            self.ema_thresholds = [
                alpha * gain + (1 - alpha) * prev_ema
                for gain, prev_ema in zip(relative_gains, self.ema_thresholds)
            ]

    def _update_sliding_baseline(self, current_losses):
        self.baseline_history.append(current_losses[:])

        self.baseline_losses = [
            sum(epoch_losses[i] for epoch_losses in self.baseline_history) / len(self.baseline_history)
            for i in range(len(current_losses))
        ]

        print("Sliding Baseline History:")
        for idx, losses in enumerate(self.baseline_history):
            print(f"Epoch {idx+1}: {losses}")
        print(f"Updated Baseline (column-wise average): {self.baseline_losses}\n")

    def _compute_relative_gains(self, curr, base):
        gains = []
        for c, b in zip(curr, base):
            diff = b - c
            denom = max(b, 1e-8)
            gain = (diff / denom) * 100.0
            print(f"curr={c:.4f}, base={b:.4f}, diff={diff:.4f}, denom={denom:.4f}, gain={gain:.4f}%")
            gains.append(gain)
        return gains


    def _evaluate_per_loss_pass_freeze(
        self, current_losses, relative_gains, stop_thresholds, epsilon=1e-2
    ):
        return [
            (gain >= threshold) or (current < epsilon)
            for current, gain, threshold in zip(current_losses, relative_gains, stop_thresholds)
        ]

    def update_weights(self, current_losses, last_qna_in_batch, in_KD=True):
        current_losses = self._to_float_list(current_losses)

        if self.baseline_losses is None:
            self.baseline_history.append(current_losses[:])
            self.baseline_losses = current_losses[:]
            self.ema_thresholds = [self.config.minimum_relative_gain] * len(current_losses)
            self.weights = self.adaptive_weighting.compute_weights(self.baseline_losses).cpu().tolist()
            print("[update_weights] Initialized baseline/EMA/weights.")
            return self.weights, False, False

        if not in_KD:
            print("[update_weights] KD disabled. Returning existing weights.")
            return self.weights, False, False

        hash_val = hashlib.md5(last_qna_in_batch.encode("utf-8")).hexdigest()

        if hash_val not in self.batch_loss_tracker_map:
            print("Hash val is initialized")
            self.batch_loss_tracker_map[hash_val] = BatchLossTracker(current_losses, self.config)

            projected_baselines = recalibrate_baselines(
                self.baseline_losses, current_losses, self.config.minimum_relative_gain
            )
            new_baseline = self.batch_loss_tracker_map[hash_val].update_baselines(projected_baselines)
        else:
            batch_tracker = self.batch_loss_tracker_map[hash_val]
            new_baseline = batch_tracker.update_baselines(current_losses)

        rel_gains_vs_baseline = self._compute_relative_gains(current_losses, new_baseline)
        rel_gains_vs_orig     = self._compute_relative_gains(current_losses, self.batch_loss_tracker_map[hash_val].og_losses)

        if len(self.batch_loss_tracker_map) > 0:
            avg_og_losses = np.mean(
                [self._to_float_list(t.og_losses) for t in self.batch_loss_tracker_map.values()],
                axis=0
            )
            avg_og_losses = self._to_float_list(avg_og_losses)
        else:
            avg_og_losses = current_losses[:]

        rel_gains_vs_avg_og = self._compute_relative_gains(current_losses, avg_og_losses)
        self.student_performance.append(rel_gains_vs_avg_og)

        per_loss_pass = self._evaluate_per_loss_pass_freeze(current_losses, rel_gains_vs_baseline, self.ema_thresholds)
        active_cutoff = self.ladder_steps[self.current_ladder_idx]
        print(f"Active cutoff: {active_cutoff}")
        print(f"Current ladder: {self.current_ladder_idx}")

        high_gain_pass = self._evaluate_per_loss_pass_freeze(
            current_losses, rel_gains_vs_orig, [active_cutoff] * self.num_losses
        )

        if all(high_gain_pass) and self.current_ladder_idx + 1 < len(self.ladder_steps):
            print(f">>> Ladder stage {active_cutoff}% passed. Advancing to {self.ladder_steps[self.current_ladder_idx+1]}% cutoff.")
            self.current_ladder_idx += 1

        freeze_condition = self._evaluate_per_loss_pass_freeze(
            current_losses, rel_gains_vs_avg_og, [self.config.relative_gain_stop_threshold] * self.num_losses
        )

        skip_kd = all(per_loss_pass) or all(high_gain_pass)
        if_freeze_student = all(freeze_condition)
        if if_freeze_student:
            skip_kd = True

        active_indices = [i for i, passed in enumerate(per_loss_pass) if not passed]

        if len(self.baseline_history) >= 5 and active_indices:
            distances = [abs(rel_gains_vs_baseline[i] - self.ema_thresholds[i]) for i in active_indices]
            tot = sum(distances)
            if tot <= 1e-12:
                normalized = [1.0 / len(active_indices)] * len(active_indices)
            else:
                normalized = [d / tot for d in distances]

            self.weights = [0.0] * self.num_losses
            for w, i in zip(normalized, active_indices):
                self.weights[i] = w
        else:
            raw = self.adaptive_weighting.compute_weights(current_losses).cpu().tolist()
            self.weights = [r if i in active_indices else 0.0 for i, r in enumerate(raw)]
            s = sum(self.weights)
            if s > 1e-12:
                self.weights = [w / s for w in self.weights]
            else:
                self.weights = [1.0 / self.num_losses] * self.num_losses

        headers = [
            "Loss#", "Current", "Baseline", "Gain_vs_EMA(%)", "EMA",
            "PerLossPass", "HighGainPass", "Gain_vs_OG(%)", "Freeze_Cond", "Weight"
        ]

        rows = [
            [i, curr, base, gain_bl, ema,
            pl_pass, hg_pass, gain_og, fr_cond, w]
            for i, (curr, base, gain_bl, ema, pl_pass, hg_pass, gain_og, fr_cond, w) in enumerate(zip(
                current_losses, new_baseline, rel_gains_vs_baseline, self.ema_thresholds,
                per_loss_pass, high_gain_pass, rel_gains_vs_orig, freeze_condition, self.weights
            ))
        ]

        print("\n----- KD & Weight Debug Info -----")
        print(tabulate(rows, headers=headers, floatfmt=".4f"))
        print(f"\nSkip KD overall? {skip_kd}")
        print("----------------------------------\n")

        return self.weights, skip_kd, if_freeze_student



    def should_skip_kd(self, current_losses_tensor, KD, update_baseline=True):
        current_losses = self._to_float_list(current_losses_tensor)

        if not KD and not update_baseline:
            info = {
                "strategy": "no-KD",
                "decision": "run",
                "reason": "KD_disabled",
                "baseline_losses": self.baseline_losses[:] if self.baseline_losses else None,
                "current_losses": current_losses,
                "relative_gains": [0.0] * len(current_losses),
                "ema_thresholds": self.ema_thresholds[:] if self.ema_thresholds else None,
                "adaptive_weights": self.weights,
                "per_loss_pass": [False] * len(current_losses)
            }
            return False, info

        if self.baseline_losses is None:
            self.baseline_history = [current_losses[:]]
            self.baseline_losses = current_losses[:]
            self.ema_thresholds = [max(0.0, self.config.minimum_relative_gain) for _ in current_losses]
            self.weights = self.adaptive_weighting.compute_weights(self.baseline_losses).cpu().tolist()
            info = {
                "strategy": "relative+EMA",
                "decision": "run",
                "reason": "initialized_baseline",
                "baseline_losses": self.baseline_losses[:],
                "current_losses": current_losses,
                "relative_gains": [0.0] * len(current_losses),
                "ema_thresholds": self.ema_thresholds[:],
                "adaptive_weights": self.weights,
                "per_loss_pass": [False] * len(current_losses)
            }
            return False, info

        self._update_sliding_baseline(current_losses)

        relative_gains = [
            ((base - curr) / max(abs(base), 1e-8)) * 100.0
            for curr, base in zip(current_losses, self.baseline_losses)
        ]

        self._update_ema_thresholds(relative_gains)
        self.ema_thresholds = [max(0.0, ema) for ema in self.ema_thresholds]

        per_loss_pass = [
            (gain >= ema) or (curr_loss < 1e-2)
            for curr_loss, gain, ema in zip(current_losses, relative_gains, self.ema_thresholds)
        ]
        skip_kd = all(per_loss_pass)

        raw_weights = self.adaptive_weighting.compute_weights(current_losses).cpu().tolist()
        masked = [rw if not pl_pass else 0.0 for rw, pl_pass in zip(raw_weights, per_loss_pass)]
        s = sum(masked)
        self.weights = [w / s for w in masked] if s > 1e-12 else [1.0 / len(masked)] * len(masked)

        if update_baseline:
            self._update_sliding_baseline(current_losses)

        info = {
            "strategy": "relative+EMA",
            "decision": "skip" if skip_kd else "run",
            "baseline_losses": self.baseline_losses[:],
            "current_losses": current_losses,
            "relative_gains": relative_gains,
            "ema_thresholds": self.ema_thresholds[:],
            "adaptive_weights": self.weights,
            "per_loss_pass": per_loss_pass
        }

        print("----- Info for Threshold Policy -----")
        for k, v in info.items():
            print(f"{k}: {v}")
        print("------------------------------------\n")

        return skip_kd, info

In [ ]:
import random

# Initialize the policy with some starting losses
initial_losses = [0.5, 0.7, 0.2]
num_losses = len(initial_losses)
policy = AdaptiveWeightedKDPolicyEMA(num_losses=num_losses, initial_losses=initial_losses, device="cpu")

current_losses = initial_losses[:]

for batch_num in range(1, 11):
    # Randomly vary current losses to simulate batch updates
    current_losses = [max(0.0, loss + random.uniform(-0.7, 0.7)) for loss in current_losses]  # bigger variation

    last_qna_in_batch = f"batch_example_qna"

    print(f"\n=== Batch {batch_num} ===")
    weights, skip_kd, freeze_student = policy.update_weights(
        current_losses=current_losses,
        last_qna_in_batch=last_qna_in_batch,
        in_KD=True
    )

    print("Batch results:")
    print("  Current losses:", [round(l, 4) for l in current_losses])
    print("  Computed weights:", [round(w, 4) for w in weights])
    print("  Skip KD:", skip_kd)
    print("  Freeze student:", freeze_student)
    print("-------------------------------------------------\n")



=== Batch 1 ===
Hash val is initialized
 Loss 0: PrevBaseline=0.5000, CurrLoss=0.4043, MinAllowed=0.3639 -> NewBaseline=0.3639 [UPDATE (pb=0.5000, cl=0.4043 -> min_allowed=0.3639)]
 Loss 1: PrevBaseline=0.7000, CurrLoss=0.8265, MinAllowed=0.7439 -> NewBaseline=0.7439 [UPDATE (pb=0.7000, cl=0.8265 -> min_allowed=0.7439)]
 Loss 2: PrevBaseline=0.2000, CurrLoss=0.1818, MinAllowed=0.1636 -> NewBaseline=0.1636 [UPDATE (pb=0.2000, cl=0.1818 -> min_allowed=0.1636)]
curr=0.4043, base=0.3639, diff=-0.0404, denom=0.3639, gain=-11.1111%
curr=0.8265, base=0.7439, diff=-0.0827, denom=0.7439, gain=-11.1111%
curr=0.1818, base=0.1636, diff=-0.0182, denom=0.1636, gain=-11.1111%
curr=0.4043, base=0.4043, diff=0.0000, denom=0.4043, gain=0.0000%
curr=0.8265, base=0.8265, diff=0.0000, denom=0.8265, gain=0.0000%
curr=0.1818, base=0.1818, diff=0.0000, denom=0.1818, gain=0.0000%
curr=0.4043, base=0.4043, diff=0.0000, denom=0.4043, gain=0.0000%
curr=0.8265, base=0.8265, diff=0.0000, denom=0.8265, gain=0.0000%

## Amateur-Expert Feedback Network

`AmateurExpertFeedbackNetwork` is designed to make a "student" model learn from a "teacher" while using expert-labeled datasets. It integrates knowledge distillation, supervised fine-tuning.


**Loss Functions Used in KD Loop:**

* **Language Modeling Loss ($L_{LM}$)**:

* **Score Regression Loss ($L_{Score}$)**:

* **Hidden Loss ($L_{hidden}$)**

* **Logits Loss ($L_{logit}$)**






* Overview

  Similar to Method 1, the KD loop also considers 5 representative losses, and loss weights are adaptively adjusted. The key diff lies in how adaptive weights and baseline losses are handled:

    * Using a single baseline or insufficient baseline data in `baseline_history` (inside `ThresholdPolicy`) can lead to extreme cases.
      
      e.g: the initial scoring loss may be very small because the student model's `scoring_head` is untrained, while other losses might be disproportionately large, makign training exhaustive. To prevent this, if the baseline loss is smaller than `self.config.minimum_relative_gain`, we instead set it to be the new baseline = `(1 - self.config.minimum_relative_gain) * curr_loss` to avoid extreme scenarios.

    * For a stable `ThresholdPolicy`, there must be at least 5 losses in `baseline_history` before the policy activates. If not, the weights are determined solely by `self.adaptive_weighting.compute_weights(curr_losses). This ensures that `baseline_losses` remain relatively plateaued and do not fluctuate a lot.



  * For the passing criterion, each loss has its own ema_threshold ([READ TO HAVE BETTER UNDERSTANDING](https://openreview.net/forum?id=2M9CUnYnBA)). Usually, ema_threshold can be negative if the model goes into extreme negative gain, but in this case, we clip the EMA to always be greater than or equal to 0 (and all cases in the ThresholdPolicy class are expressed in percentage terms). The adaptive_weights are not determined like above ([Method 1](#method-1)) but more like:

  adaptive_weights = (distance between EMA threshold and relative gains) / (sum of the distances).

  If all five losses pass the threshold criteria, and the loop is before the determined exit, it can still be skipped to avoid overtraining.

    

In [ ]:
class LossConfig:
    def __init__(self, enabled_flags):
        self.loss_names = ["lm_loss", "hidden_loss", "scoring_loss", "logit_loss"]

        self.scales = {
            "lm_loss" : 1.0,
            "hidden_loss" : 1.0,
            "scoring_loss" : 1.0,
            "logit_loss" : 0.03
        }

        if len(enabled_flags) != len(self.loss_names):
            raise ValueError(f"Expected {len(self.loss_names)} flags, got {len(enabled_flags)}")

        self.enabled = dict(zip(self.loss_names, enabled_flags))
        self.num_enabled = sum(enabled_flags)

    def get_active_losses(self):
        active = [name for name, flag in self.enabled.items() if flag]
        return active, self.enabled

    def toggle_loss(self, loss_name, state):
        if loss_name not in self.enabled:
            raise KeyError(f"{loss_name} not a valid loss. Choose from {self.loss_names}.")
        self.enabled[loss_name] = state
        self.num_enabled = sum(self.enabled.values())

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from sentence_transformers import SentenceTransformer
import re
from difflib import SequenceMatcher
import random


class AmateurExpertFeedbackNetWork:
    def __init__(self, student, teacher, expert_datasets=None, loss_flags=None, device=None, optimizer=None):
        self.student = student
        self.teacher = teacher
        self.expert_datasets = expert_datasets or []
        self.student_tokenizer = student.tokenizer
        self.teacher_tokenizer = teacher.tokenizer
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.student.score_head = self.student.score_head.to(torch.float32)
        self.model_dtype = torch.float32
        self.loss_flags = [True, True, True, True] if loss_flags is None else loss_flags
        self.loss_config = LossConfig(self.loss_flags)

        self.threshold_policy = AdaptiveWeightedKDPolicyEMA(num_losses = self.loss_config.num_enabled)

        for tokenizer in [self.student_tokenizer, self.teacher_tokenizer]:
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

        self.teacher.model.eval()
        with torch.no_grad():
            dummy_text = "Hello World"
            student_input = self.student_tokenizer(dummy_text, return_tensors="pt").to(self.device)
            teacher_input = self.teacher_tokenizer(dummy_text, return_tensors="pt").to(self.device)

            student_out = self.student.model(**student_input, output_hidden_states=True, return_dict=True)
            teacher_out = self.teacher.model(**teacher_input, output_hidden_states=True, return_dict=True)

            student_hidden_dim = student_out.hidden_states[-1].size(-1)
            teacher_hidden_dim = teacher_out.hidden_states[-1].size(-1)

        if student_hidden_dim != teacher_hidden_dim:
            self.align_hidden_for_hidden_loss = nn.Linear(student_hidden_dim, teacher_hidden_dim)
        else:
            self.align_hidden_for_hidden_loss = nn.Identity()

        self.align_hidden_for_hidden_loss = self.align_hidden_for_hidden_loss.to(self.device).to(self.model_dtype)

        same_vocab = (
            self.student.model.config.vocab_size == self.teacher.model.config.vocab_size
            and self.student_tokenizer.get_vocab() == self.teacher_tokenizer.get_vocab()
        )

        self.projection_layer = nn.Identity()

        if not same_vocab:
            print("[Logit KD] Different tokenizers/vocabs detected → disabling logit_loss.")
            try:
                self.loss_config.toggle_loss("logit_loss", False)
            except Exception:
                pass

        self.optimizer = AdamW(
            [
                {"params" : self.student.model.parameters(), "lr" : 5e-7, "weight_decay" : 0.01},
                {"params": self.student.score_head.parameters(), "lr": 5e-5, "weight_decay": 0.01},
                {"params": self.align_hidden_for_hidden_loss.parameters(), "lr": 1e-6, "weight_decay": 0.01},
                {"params": self.projection_layer.parameters(), "lr": 1e-6, "weight_decay": 0.01},
            ],
            betas=(0.9, 0.999),
            eps=1e-8
        )

    def knowledge_distillation_with_SFT(self, weights, batch_size=3, epochs=3, stopped=20, is_math = False, examples=None):
        random.shuffle(self.expert_datasets)

        loss_names, _ = self.loss_config.get_active_losses()
        loss_history = {loss_name: [] for loss_name in loss_names}

        total_steps = epochs * min(stopped, len(self.expert_datasets))
        global_step = 0

        for epoch in range(epochs):
            print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

            epoch_losses = {name: 0.0 for name in loss_names}
            epoch_batch_counts = {name: 0 for name in loss_names}

            num_samples = 0
            stop_epoch = False
            batches_in_epoch = 0

            last_question_and_answer_in_batch = ""
            for batch_start in range(0, len(self.expert_datasets), batch_size):
                batch = self.expert_datasets[batch_start: batch_start + batch_size]
                if stopped is not None and num_samples >= stopped:
                    break

                each_loss_in_batch = {loss_name: [] for loss_name in loss_names}

                for sample in batch:
                    prompt_text = sample.get("prompt")
                    answer = sample.get("answer")
                    expert_score = sample.get("score")
                    expert_feedback = sample.get("feedback")

                    if not prompt_text or not answer or expert_score is None:
                        continue

                    print(f"\nSample {num_samples + 1}")
                    student_feedback, student_score_value, student_score_logit = self.student.generate_feedback(
                        prompt_text, answer, is_math, examples
                    )
                    with torch.no_grad():
                        teacher_feedback, teacher_score = self.teacher.generate_feedback(
                            prompt_text, answer, is_math, examples
                        )

                    if "lm_loss" in loss_names:
                        lm_loss = compute_lm_loss(teacher_feedback, self.student, prompt_text, answer,
                                                  self.device, self.model_dtype, is_math, examples)
                        each_loss_in_batch["lm_loss"].append(lm_loss)

                    if "hidden_loss" in loss_names:
                        hidden_loss = compute_hidden_loss(teacher_feedback, self.student_tokenizer, self.teacher_tokenizer,
                                                          self.student, self.teacher, self.align_hidden_for_hidden_loss,
                                                          self.device, self.model_dtype)
                        each_loss_in_batch["hidden_loss"].append(hidden_loss)

                    if "scoring_loss" in loss_names:
                        score_loss = compute_scoring_loss(student_score_logit, teacher_score,
                                                          self.model_dtype, self.device)
                        each_loss_in_batch["scoring_loss"].append(score_loss)

                    if "logit_loss" in loss_names:
                        logit_loss = compute_logit_standardization(self.student, self.teacher,
                                                                  self.student_tokenizer, self.teacher_tokenizer,
                                                                  self.projection_layer, expert_feedback)
                        each_loss_in_batch["logit_loss"].append(logit_loss)

                    num_samples += 1
                    last_question_and_answer_in_batch = prompt_text + answer

                batch_losses = {
                    name: torch.stack(vals).mean()
                    for name, vals in each_loss_in_batch.items() if len(vals) > 0
                }
                if not batch_losses:
                    continue

                curr_losses_detached = [batch_losses[name].detach() for name in loss_names if name in batch_losses]

                if global_step == 0:
                    assigned_weights = weights
                    skip_kd = False
                    if_freeze_student = False
                else:
                    assigned_weights, skip_kd, if_freeze_student = self.threshold_policy.update_weights(
                        curr_losses_detached, last_question_and_answer_in_batch
                    )

                loss_weight_map = dict(zip(loss_names, assigned_weights))
                weights_str = ", ".join([f"{name}={w:.4f}" for name, w in loss_weight_map.items()])
                print(f"[Adaptive Weights] {weights_str}, skip_kd={skip_kd}")

                total_loss = 0.0
                for name, w in loss_weight_map.items():
                    if name in batch_losses and batch_losses[name] is not None:
                        total_loss = total_loss + w * self.loss_config.scales.get(name, 1.0) * batch_losses[name]

                self.optimizer.zero_grad()
                total_loss.backward()
                self.optimizer.step()

                for name, loss_val in batch_losses.items():
                    epoch_losses[name] += float(loss_val.item())
                    epoch_batch_counts[name] += 1

                for name, loss_val in batch_losses.items():
                    loss_history[name].append(float(loss_val.item()))

                print("---- FEEDBACK COMPARISON ----")
                print(f"[Student Feedback]: {student_feedback}")
                print(f"[Teacher Feedback]: {teacher_feedback}")
                print(f"[Expert  Feedback]: {expert_feedback}")
                print("---- LOSS VALUES ----")
                for name, batch_loss in batch_losses.items():
                    print(f"{name}: {batch_loss.item():.4f}")
                print(f"TOTAL LOSS: {total_loss.item():.4f}")
                print("------------------------------\n")
                for name, batch_loss in batch_losses.items():
                    print(f"{name} requires_grad: {batch_loss.requires_grad}")

                global_step += 1
                batches_in_epoch += 1

                if skip_kd:
                    print("---- SKIPPING KD ----")
                    print(f"Epoch: {epoch + 1}/{epochs}, Iteration: {batch_start // batch_size + 1}, Global step: {global_step}")
                    print("Batch Losses:", ", ".join([f"{n}={batch_losses[n].item():.4f}" for n in batch_losses]))
                    print(f"Adaptive weights: {assigned_weights}")
                    print("--------------------\n")
                    stop_epoch = True
                    if if_freeze_student:
                        print(">>> FREEZING STUDENT MODEL (all stop-thresholds satisfied) <<<")
                        self.student.freeze_student_model()
                    break

            epoch_avgs = {
                n: (epoch_losses[n] / max(1, epoch_batch_counts[n]))
                for n in epoch_losses
            }
            print(f"Epoch {epoch + 1} averages:", ", ".join([f"{n}={v:.4f}" for n, v in epoch_avgs.items()]))

            if stop_epoch:
                print(f"[Info] KD skipping triggered. Ending epoch {epoch + 1} early.")
                break

        visualize_student_performance(
            self.threshold_policy.student_performance,
            self.threshold_policy.config.relative_gain_stop_threshold,
            loss_names
        )

        final_loss_vector = epoch_avgs

        return {
            "teacher": self.teacher,
            "student": self.student,
            "loss_history": loss_history,
            "final_loss_vector": final_loss_vector,
            "epochs_ran": epochs,
            "samples_per_epoch": stopped,
        }


    def _fuse_scores(self, student_score, teacher_score, threshold):
        return float((threshold * student_score +  teacher_score) / (1 + threshold))

    def generate_combined_feedback(
        self,
        prompt_text,
        answer,
        is_math = False,
        examples = None,
        epochs=1,
        iterations=20,
        threshold=0.65,
        KD=True,
        update_baseline = True
    ):
        result = compute_all_losses_vectorized(
            self.loss_config,
            prompt_text, answer,
            self.student, self.teacher,
            self.student_tokenizer, self.teacher_tokenizer,
            self.align_hidden_for_hidden_loss, self.projection_layer,
            device=self.device,
            is_math=is_math, examples=examples,
            model_dtype=self.model_dtype
        )

        student_feedback, teacher_feedback, student_score, teacher_score = unpack_feedback_and_scores(result)

        is_skipping, info = self.threshold_policy.should_skip_kd(result['loss_vector'], KD, update_baseline)
        adaptive_weights = info['adaptive_weights']
        if self.student.student_frozen:
            print("[Info] Student model already frozen. Skipping distillation.")
            return {
                "expert_feedback": teacher_feedback,
                "amateur_feedback": student_feedback,
                "combined_score": (threshold * student_score +  teacher_score) / (1 + threshold),
                "loss_vector": result["loss_vector"]
            }

        if is_skipping:
            print("[Info] Skipping KD for this sample based on threshold criteria.")
            return {
                "expert_feedback": teacher_feedback,
                "amateur_feedback": student_feedback,
                "combined_score": self._fuse_scores(student_score, teacher_score, threshold),
                "loss_vector": result["loss_vector"]
            }

        if not KD:
            return {
                "expert_feedback": teacher_feedback,
                "amateur_feedback": student_feedback,
                "combined_score": self._fuse_scores(student_score, teacher_score, threshold),
                "loss_vector": result["loss_vector"]
            }
        else:
            print("[Distill Trigger] Initiating knowledge distillation...")

            self.knowledge_distillation_with_SFT(weights=adaptive_weights, epochs=epochs, stopped=iterations, is_math=is_math, examples=examples)

            updated_result = compute_all_losses_vectorized(
                self.loss_config,
                prompt_text, answer,
                self.student, self.teacher,
                self.student_tokenizer, self.teacher_tokenizer,
                self.align_hidden_for_hidden_loss,self.projection_layer, device=self.device, is_math=is_math, examples = examples
            )

            new_loss_vector = updated_result["loss_vector"].unsqueeze(0)
            student_feedback, teacher_feedback, student_score, teacher_score = unpack_feedback_and_scores(updated_result)

        is_skipping, info = self.threshold_policy.should_skip_kd(updated_result['loss_vector'], KD = False, update_baseline=False)
        adaptive_weights = info['adaptive_weights']
        combined_score = self._fuse_scores(student_score, teacher_score, threshold)

        return {
            "expert_feedback": teacher_feedback,
            "amateur_feedback": student_feedback,
            "combined_score": combined_score,
            "loss_vector": updated_result["loss_vector"]
        }

In [ ]:
loss_flags = [True, True, True, True]
network = AmateurExpertFeedbackNetWork(amateur_feedback_model, expert_model, expert_datasets, loss_flags=loss_flags)
expert_model.network = network

**A debugging JSON file for KD network** -> [feedback_logs](https://drive.google.com/file/d/1TjuuyMqx3WFs3xLBVHPlG9ugb2A1LVoJ/view?usp=sharing)

## Parsing Model

This `ParsingModel` model is used to extract the final numerical answer from a given text answer. The goal is to isolate just the numerical value - whether an integer, decimal, fraction while ignoring words or units.



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

def truncate_to_first_paragraph(text):
    for part in text.strip().split("\n\n"):
        if part.strip():
            return part.strip()
    return text.strip()

class ParsingModel:
    def __init__(self, tokenizer, model, model_name):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.tokenizer = tokenizer
        self.model = model
        print(f"Loaded {model_name} on {self.device}")

        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate_prompt(self, answer, is_math=False):
      if is_math:
          prompt = (
              "You are a mathematical answer extractor tasked with extracting the final numerical answer "
              "from a complicated formatted answer. Extract only the numerical value, which can be an integer, "
              "decimal, fraction (in a/b form), or percentage. Keep fractions in a/b format and keep % if present. "
              "Do not include any words, units, or additional context. Think carefully before responding. Format your output as follows:\n\n"
              "Extracted final numerical answer: <numerical value>\n\n"
              "Answer: The total cost is $120, plus tax, so \\boxed{120} is the amount before tax.\n"
              "Extracted final numerical answer: 120\n\n"
              f"Answer: {answer}\n"
              "Extracted final numerical answer:"
          )

      else:
          prompt = (
                "You are an answer extractor. From the given Answer text, copy the SHORTEST CONTIGUOUS SPAN "
                "that directly answers the question. Do NOT invent text. Output ONLY that span.\n\n"
                "Formatting:\n"
                "- Print exactly one line: 'Extracted final answer: <span>'\n"
                "- No extra words, no explanations, no quotes.\n\n"
                "Rules for difficult cases:\n"
                "1) DATES: If a month name (or abbreviation) appears anywhere in the Answer with a matching day, "
                "   prefer the full date phrase (e.g., 'September 23rd') over a bare day ('23'). Keep ordinals.\n"
                "2) NAMED ENTITIES WITH DETERMINERS: If the Answer’s concluding statement uses a determiner with "
                "   a proper noun (e.g., 'the Nile', 'the Netherlands'), KEEP it. For people (e.g., 'Abraham Lincoln'), "
                "   just the name.\n"
                "3) NUMBERS: If the final answer is a number, return just the number.\n"
                "4) COPY EXACTLY from the provided Answer text; do not normalize, translate, or change casing.\n\n"
                "Answer: The passage states that the capital of France is Paris, after discussing other cities.\n"
                "Extracted final answer: Paris\n\n"
                f"Answer: {answer}\n"
                "Extracted final answer:"
          )

      return prompt

    def generate_answer(self, answer, is_math = False, max_new_tokens=50):
        full_prompt = self.generate_prompt(answer, is_math)

        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id
        )

        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        extracted = generated_text[len(full_prompt):].strip()

        return truncate_to_first_paragraph(extracted)

KeyboardInterrupt: 

In [ ]:
parse_model = ParsingModel(teacher_tokenizer, teacher_model, teacher_model_name)

# General Data Loading Setup


In [ ]:
import csv
import os
import json

def save_results_to_csv(result, features, file_path):
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


def converting_from_csv_to_json(features, csv_path, json_path):
    results = []
    with open(csv_path, 'r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)

        for row in csv_reader:

            result = {feature: value for feature, value in zip(features, row)}
            results.append(result)

    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(results, json_file, indent=4, ensure_ascii=False)

In [ ]:
def reloading_result(file_path):
    with open(file_path, 'r') as file:
      results = json.load(file)

    return results

## Main Experiment Loop

**Experiment metrics and utilities**

* **Cosine Similarity between Feedbacks**:

  It compares the textual feedback from the student and expert models using a sentence embedding model (`all-MiniLM-L6-v2`). The similarity betweent their feedback is measured using cosine similarity, helping us understand how closely the student's responses align with the expert's guidance

* **Prediction Accuracy**:

  For each task, the script checks whether the model produces the correct final answer. This allows us to track how accurate the student model becomes over time or iterations.

* **Toxicity Analysis**:

  The script also measures the toxicity levels of different types of feedback: expert's original response, the student's version, and any refined responses. These scores are plotted to compare how safe or potentially harmful each type of feedback is

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
import re
from detoxify import Detoxify

def compute_similarity(fb1, fb2):
    vec1 = torch.tensor(embedder.encode([fb1]), dtype=torch.float32)
    vec2 = torch.tensor(embedder.encode([fb2]), dtype=torch.float32)

    cos_sim = F.cosine_similarity(vec1, vec2)

    return cos_sim.item()

def extract_final_answer(text):
    match = re.search(r"####\s*(-?\d+(?:\.\d+)?)", text)
    if match:
        num_str = match.group(1)
        return float(num_str) if "." in num_str else int(num_str)
    return None

def plot_experiment_metrics(similarities, accuracies, save_path=None):
    tasks = list(range(1, len(similarities) + 1))

    plt.figure(figsize=(12, 6))

    # Plot cosine similarity
    plt.subplot(1, 2, 1)
    plt.plot(tasks, similarities, marker='o', color='blue', label='Cosine Similarity')
    plt.title('Cosine Similarity Between Feedbacks')
    plt.xlabel('Task Number')
    plt.ylabel('Cosine Similarity')
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend()

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(tasks, accuracies, marker='x', color='green', label='Prediction Accuracy')
    plt.title('Prediction Accuracy Over Tasks')
    plt.xlabel('Task Number')
    plt.ylabel('Accuracy')
    plt.ylim(-0.1, 1.1)
    plt.grid(True)
    plt.legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        print(f"Plot saved to: {save_path}")

    plt.show()


def plot_toxicities(expert_toxicities, amateur_toxicities, refined_toxicities, save_path):
    plt.figure(figsize=(12, 6))
    x = range(1, len(expert_toxicities) + 1)

    plt.plot(x, expert_toxicities, label="Expert Feedback Toxicity", color="blue", marker="o")
    plt.plot(x, amateur_toxicities, label="Amateur Feedback Toxicity", color="green", marker="^")
    plt.plot(x, refined_toxicities, label="Refined Answer Toxicity", color="red", marker="s")

    plt.xlabel("Task Index")
    plt.ylabel("Toxicity Score")
    plt.title("Toxicity Scores per Task")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Toxicity plot saved to: {save_path}")

def plot_metrics(bert_scores, bleu_scores, rouge_scores, save_path):
    plt.figure(figsize=(10, 6))
    plt.plot(bert_scores, label="BERTScore F1")
    plt.plot(bleu_scores, label="BLEU")
    plt.plot(rouge_scores, label="ROUGE-L F")
    plt.xlabel("Sample Index")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics for Refined Answers")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()

### Experiment for gsm8k dataset

**Features of `gsm8k` Dataset**

* `question`: A math word problem in natural language.

* `answer`: A step-by-step solution ending with the final numeric answer, typically marked by a #### tag.

**Example GSM8K Instance**:

`{
    'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
    'answer': 'How many clips did Natalia sell in May? ** Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nHow many clips did Natalia sell altogether in April and May? ** Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72',
}
`


**Purpose of `run_experiment_gsm8k(...)`**:

This function runs an experiment to evaluate how well an expert model improves answers using both expert and amateur feedback, and measures:

* Accuracy of final answers

* Similarity between feedbacks

* Toxicity of feedbacks and refined answers

* Visualization of results

In [ ]:
from datasets import load_dataset

gsm8k_ds = load_dataset("openai/gsm8k", "main")

train_dataset = gsm8k_ds['train']

split = train_dataset.train_test_split(test_size=0.1, seed=42)
gsm8k_train_dataset = split['train']
gsm8k_test_dataset = split['test']

In [ ]:
import os
import re
from detoxify import Detoxify

def run_experiment_gsm8k(dataset, dataset_name, expert_model, feedback_network=None, task_description="Solve the following math problem step-by-step", is_math=True, examples = math_few_shot_examples):
    csv_path = f"{dataset_name}_experiment_results.csv"

    results = []
    similarities = []
    accuracies = []

    expert_feedback_toxicities = []
    amateur_feedback_toxicities = []
    refined_answer_toxicities = []

    for i, sample in enumerate(dataset):
        if i >= 200:
            break

        gsm8k_input_prompt = sample["question"]
        ground_truth = extract_final_answer(sample["answer"])

        gsm8k_initial_output = expert_model.generate_answer(gsm8k_input_prompt, is_math, task_description)

        refined_answer, expert_feedback, amateur_feedback, unified_feedback = expert_model.improve_answer_with_feedback_and_critique(
            gsm8k_input_prompt, gsm8k_initial_output, epochs=2, iterations=10, is_math=is_math, examples = examples
        )

        cosine_sim = compute_similarity(expert_feedback, amateur_feedback)

        gsm8k_pred_answer = parse_model.generate_answer(refined_answer, is_math=is_math)
        gsm8k_pred_answer = gsm8k_pred_answer.replace(",", "").strip()
        gsm8k_pred_answer = re.sub(r"[^\d\.\-]", "", gsm8k_pred_answer)

        pred_answer = None
        try:
            pred_answer = int(gsm8k_pred_answer)
        except ValueError:
            try:
                pred_answer = float(gsm8k_pred_answer)
            except ValueError:
                pred_answer = None

        accuracy = int(ground_truth == pred_answer) if pred_answer is not None else 0

        similarities.append(cosine_sim)
        accuracies.append(accuracy)

        print(f"\n--- Task {i+1} ---")
        print(f"Prompt: {gsm8k_input_prompt}")
        print(f"x0 (Initial Output): {gsm8k_initial_output}")
        print(f"x1 (Refined Answer): {refined_answer}")
        print("Expert Feedback:", expert_feedback)
        print("Amateur Feedback:", amateur_feedback)
        print("Similarity:", round(cosine_sim, 3))
        print("Prediction:", pred_answer, "| Ground Truth:", ground_truth)
        print("Correct:", bool(accuracy))

        try:
            expert_tox = Detoxify("original").predict(expert_feedback)["toxicity"]
        except Exception as e:
            print(f"[Error] Expert Feedback Toxicity failed: {e}")
            expert_tox = None

        try:
            amateur_tox = Detoxify("original").predict(amateur_feedback)["toxicity"]
        except Exception as e:
            print(f"[Error] Amateur Feedback Toxicity failed: {e}")
            amateur_tox = None

        try:
            refined_tox = Detoxify("original").predict(refined_answer)["toxicity"]
        except Exception as e:
            print(f"[Error] Refined Answer Toxicity failed: {e}")
            refined_tox = None

        expert_feedback_toxicities.append(expert_tox)
        amateur_feedback_toxicities.append(amateur_tox)
        refined_answer_toxicities.append(refined_tox)

        result = {
            "question": gsm8k_input_prompt,
            "x0": gsm8k_initial_output,
            "expert_feedback": expert_feedback,
            "amateur_feedback": amateur_feedback,
            "similarity": cosine_sim,
            "x1": refined_answer,
            "pred": pred_answer,
            "label": ground_truth,
            "correct": bool(accuracy)
        }

        results.append(result)

        save_results_to_csv(
            result=result,
            features=["question", "x0", "expert_feedback", "amateur_feedback", "similarity", "x1", "pred", "label", "correct"],
            file_path=csv_path
        )

    plot_experiment_metrics(
        similarities,
        accuracies,
        save_path=f"{csv_dir}/{dataset_name}_similarity_accuracy_plot_main_gsm8k.png"
    )

    plot_toxicities(
        expert_feedback_toxicities,
        amateur_feedback_toxicities,
        refined_answer_toxicities,
        save_path=f"{csv_dir}/{dataset_name}_toxicity_plot_main_gsm8k.png"
    )

    print(f"\nExperiment complete. Results saved to: {csv_path}")
    return results

**gsm8k experiment**

In [ ]:
gsm8k_experiment_result = run_experiment_gsm8k(gsm8k_test_dataset, "gsm8k", expert_model, network)

### Experiment for `drop` dataset

**Features of `drop` dataset**:

* `section_id`: identifier for the document section

* `query_id` : unique id for the question/query

* `passage` : the passage of text containing the information

* `question` : the question posed about the passage  

* `answers_spans` : the answer(s) as text spans within the passage


**Example `drop` instance**:


`{
  "section_id": "section_1234",
  "query_id": "query_5678",
  "passage": "The Empire State Building was completed in 1931 and stands at 1,454 feet tall.",
  "question": "When was the Empire State Building completed?",
  "answers_spans": ["1931"]
}`

**Purpose of `run_experiment_for_drop(...)`**:

  This funcion runs an experiment on a WTB-style dataset to test how well an expert model can refine answers using feedback, and then measures the model's accuracy, feedback similarity, and toxicity levels.

In [ ]:
from datasets import load_dataset

drop_ds = load_dataset("ucinlp/drop")
train_dataset = drop_ds["train"]
split = train_dataset.train_test_split(test_size=0.1, seed=42)
drop_train_ds = split["train"]
drop_test_ds  = split["test"]

drop_test_ds_clone = []
for ex in drop_test_ds:
    passage  = ex.get("passage", "")
    question = ex.get("question", "")

    spans = ex.get("answers_spans", [])
    labels = spans.get("spans")

    drop_test_ds_clone.append({
        "passage": passage,
        "question": question,
        "answer": labels,
    })

In [ ]:
import os
import re

import re

def _normalize_answer(s):
    s = s.strip().lower()
    s = re.sub(r'(?<=\d)[,\s](?=\d)', '', s)
    s = re.sub(r'\s+', ' ', s)

    s = re.sub(r'[.,;:\s]+$', '', s)

    return s


_num_pat = re.compile(r'^[-+]?\d+(?:\.\d+)?%?$')

def _canon_number(s):
    t = re.sub(r'(?<=\d)[,\s](?=\d)', '', s.strip().lower())
    return t

def _is_correct(pred, gt):
    if pred is None or gt is None:
        return 0

    pred_n = _normalize_answer(pred)

    def _match(one, two):
        a = _normalize_answer(one)
        b = _normalize_answer(two)
        if _num_pat.match(a) and _num_pat.match(b):
            return _canon_number(a) == _canon_number(b)
        return a == b

    if isinstance(gt, (list, tuple, set)):
        return int(any(_match(pred, x) for x in gt))
    return int(_match(pred, gt))


def _safe_toxic(detox, text):
    if detox is None or text is None:
        return None
    try:
        return detox.predict(text).get("toxicity")
    except Exception:
        return None

def run_experiment_for_drop(
    dataset,
    dataset_name,
    expert_model,
    feedback_network,
    max_iters=25,
    csv_dir="./main_results",
    task_description="Identify a book based on a vague reader description.",
    detox=None
):
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f"{csv_dir}/{dataset_name}_experiment_results.csv"

    drop_features = [
        "drop_x0", "drop_expert_feedback", "drop_amateur_feedback",
        "drop_similarity", "drop_x1", "drop_pred", "drop_label", "drop_correct"
    ]

    drop_results = []
    drop_similarities = []
    drop_accuracies = []
    drop_expert_toxicities = []
    drop_amateur_toxicities = []
    drop_refined_toxicities = []

    for i, sample in enumerate(dataset):
        if i >= max_iters:
            break

        drop_passage = sample.get("passage")
        drop_question = sample.get("question")
        drop_label = sample.get("answer")

        drop_input_prompt = f"{drop_passage}\nQuestion: {drop_question}"

        drop_x0 = expert_model.generate_answer(drop_input_prompt, task_description=task_description)

        drop_x1, drop_expert_feedback, drop_amateur_feedback, drop_unified_feedback = (
            expert_model.improve_answer_with_feedback_and_critique(
                drop_input_prompt, drop_x0, epochs=2, iterations=10
            )
        )

        drop_similarity = compute_similarity(drop_expert_feedback, drop_amateur_feedback)
        drop_pred_answer = parse_model.generate_answer(drop_x1)
        drop_correct = _is_correct(drop_pred_answer, drop_label)

        drop_expert_toxic = _safe_toxic(detox, drop_expert_feedback)
        drop_amateur_toxic = _safe_toxic(detox, drop_amateur_feedback)
        drop_refined_toxic = _safe_toxic(detox, drop_x1)

        drop_expert_toxicities.append(drop_expert_toxic)
        drop_amateur_toxicities.append(drop_amateur_toxic)
        drop_refined_toxicities.append(drop_refined_toxic)

        drop_similarities.append(drop_similarity)
        drop_accuracies.append(drop_correct)

        print(f"\n--- drop_Task {i+1} ---")
        print(f"drop_prompt: {drop_input_prompt}")
        print(f"drop_x0 (initial): {drop_x0}")
        print(f"drop_p_expert: {drop_expert_feedback}")
        print(f"drop_p_student: {drop_amateur_feedback}")
        print("drop_similarity:", round(drop_similarity, 3))
        print(f"drop_x1 (refined): {drop_x1}")
        print("drop_pred:", drop_pred_answer, "| drop_label(s):", drop_label)
        print("drop_correct:", bool(drop_correct))

        drop_results.append({
            "drop_x0": drop_x0,
            "drop_expert_feedback": drop_expert_feedback,
            "drop_amateur_feedback": drop_amateur_feedback,
            "drop_similarity": drop_similarity,
            "drop_x1": drop_x1,
            "drop_pred": drop_pred_answer,
            "drop_label": drop_label,
            "drop_correct": bool(drop_correct),
        })

    save_results_to_csv(drop_results, drop_features, csv_path)
    plot_experiment_metrics(
        drop_similarities, drop_accuracies,
        save_path=f"{csv_dir}/{dataset_name}_similarity_accuracy_plot_main_drop.png"
    )
    plot_toxicities(
        drop_expert_toxicities, drop_amateur_toxicities, drop_refined_toxicities,
        save_path=f"{csv_dir}/{dataset_name}_toxicity_plot_main_drop.png"
    )

    print(f"\nExperiment complete. Results saved to: {csv_path}")
    return drop_results

**drop experiment**

In [ ]:
drop_experiment_result = run_experiment_for_drop(drop_test_ds_clone, "drop", expert_model, network)

### Experiment for alpaca dataset

**Features of `alpaca` Dataset**

* `instruction`: describes the task the model should perform. Each of the 52K instructions is unique

* `input`: optional context or input for the task. For example, when the instruction is "Summarize the following article", the input is the article. Around 40% of the examples have an input

* `output`: the answer to the instruction as generated by `text-davinci-003`

* `text`: the `instruction`, `input`, `and `output` formatted with the prompt template used by the authors for fine-tuning their models

**Example alpaca Instance**:

`{
    "instruction": "Create a classification task by clustering the given list of items.",
    "input": "Apples, oranges, bananas, strawberries, pineapples",
    "output": "Class 1: Apples, Oranges\nClass 2: Bananas, Strawberries\nClass 3: Pineapples",
    "text": "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a classification task by clustering the given list of items.\n\n### Input:\nApples, oranges, bananas, strawberries, pineapples\n\n### Response:\nClass 1: Apples, Oranges\nClass 2: Bananas, Strawberries\nClass 3: Pineapples",
}
`


**Purpose of `run_experiment_for_alpaca(...)`**:

This functions runs a performance and feedback evaluation on Alpaca using:

  * An expert model
  * A feedback mechanism
  * Several evaluation metrics like mentioned above to assess the refinement process

In [ ]:
from datasets import load_dataset

alpaca_ds = load_dataset("tatsu-lab/alpaca")

train_dataset = alpaca_ds['train']

split = train_dataset.train_test_split(test_size=0.1, seed=42)
alpaca_train_dataset = split['train']
alpaca_test_dataset = split['test']

In [ ]:
import os
import matplotlib.pyplot as plt
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def run_experiment_for_alpaca(dataset, dataset_name, expert_model, feedback_network, max_iters=1, csv_dir="./results"):
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f"{csv_dir}/{dataset_name}_experiment_results.csv"

    results = []
    bert_scores = []
    bleu_scores = []
    rouge_l_scores = []
    cosine_similarities = []

    expert_toxicities = []
    amateur_toxicities = []
    refined_toxcities = []

    scorer = rouge_scorer.RogueScorer(["rougeL"], use_stemmer = True)
    detox = Detoxify("original")

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    for i, sample in enumerate(dataset):
        instruction = sample["instruction"]
        input_text = sample.get("input", "").strip()
        ground_truth = sample["output"].strip()

        input_prompt = input_text

        initial_output = expert_model.generate_answer(input_prompt, task_description = instruction)

        refined_answer, expert_feedback, amateur_feedback, unified_feedback = (
            expert_model.improve_answer_with_feedback_and_critique(input_prompt, initial_output)
        )

        cosine_sim = compute_similarity(expert_feedback, amateur_feedback)
        cosine_similarities.append(cosine_sim)

        expert_tox = detox.predict(expert_feedback)["toxicity"]
        amateur_tox = detox.predict(amateur_feedback)["toxicity"]
        refined_tox = detox.predict(refined_answer)["toxicity"]
        expert_toxicities.append(expert_tox)
        amateur_toxicities.append(amateur_tox)
        refined_toxicities.append(refined_tox)

        P, R, F1 = bert_score([refined_answer], [ground_truth], lang="en", verbose=False)
        bert_f1 = F1[0].item()
        bert_scores.append(bert_f1)

        bleu = sentence_bleu(
            [ground_truth.split()],
            refined_answer.split(),
            smoothing_function=SmoothingFunction().method1
        )
        bleu_scores.append(bleu)

        rouge_result = scorer.score(ground_truth, refined_answer)
        rouge_l = rouge_result["rougeL"].fmeasure
        rouge_l_scores.append(rouge_l)

        print(f"\n--- Task {i+1} ---")
        print(f"Prompt: {input_prompt}")
        print(f"x0 (Initial output): {initial_output}")
        print(f"p_expert: {expert_feedback}")
        print(f"p_student: {amateur_feedback}")
        print("Similarity:", round(cosine_sim, 3))
        print(f"x1 (Refined Answer): {refined_answer}")

        result = {
            "x0": initial_output,
            "expert_feedback": expert_feedback,
            "amateur_feedback": amateur_feedback,
            "similarity": cosine_sim,
            "x1": refined_answer,
            "label": ground_truth,
        }

        results.append(result)
        save_results_to_csv(
            result=result,
            features=["x0", "expert_feedback", "amateur_feedback", "similarity", "x1", "label"],
            file_path=csv_path
        )

    plot_metrics(
        bert_scores, bleu_scores, rouge_l_scores, cosine_similarities,
        save_path=f"{csv_dir}/{dataset_name}_metrics_plot_main_alpaca.png"
    )

    plot_toxicities(
      expert_toxicities, amateur_toxicities, refined_toxicities,
      save_path=f"{csv_dir}/{dataset_name}_toxicity_plot_main_alpaca.png"
    )

    print(f"\nExperiment complete. Results saved to: {csv_path}")
    return results

**alpaca experiment**

In [ ]:
alpaca_experiment_result = run_experiment(alpaca_test_dataset, "wtb", expert_model, network)

## agieval math

In [ ]:
from datasets import load_dataset
import re

ds = load_dataset("hails/agieval-math")
agieval_math_ds_test = ds["test"]

def sanitize_question(question):
    sanitized_question = re.sub(r"^Q:\s*", "", question)
    return sanitized_question.strip()

def clean_answer_in_prompt(answer):
    sanitized_answer = re.sub(r"\s*A:\s*The\s+answer\s+is\s*$", "", answer)
    return sanitized_answer.strip()

sanitized_data = []

for sample in agieval_math_ds_test:
    question = sample["query"]
    answer = sample["answer"]

    sanitized_question = sanitize_question(question)
    sanitized_question = clean_answer_in_prompt(sanitized_question)

    sanitized_data.append({
        "query": sanitized_question,
        "answer": answer
    })


In [ ]:
import os
import re
from detoxify import Detoxify

def run_experiment_agieval_math(dataset, dataset_name, expert_model, feedback_network=None, csv_dir="./main_results", task_description="Solve the following math problem step-by-step", is_math=True, examples=math_few_shot_examples):
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f"{csv_dir}/{dataset_name}_experiment_results.csv"

    results = []
    similarities = []
    accuracies = []

    expert_feedback_toxicities = []
    amateur_feedback_toxicities = []
    refined_answer_toxicities = []

    for i, sample in enumerate(dataset):
        if i >= 20:
            break

        agieval_math_input_prompt = sample["query"]
        ground_truth = extract_final_answer(sample["answer"])

        agieval_math_initial_output = expert_model.generate_answer(agieval_math_input_prompt, is_math, task_description, examples)

        refined_answer, expert_feedback, amateur_feedback, unified_feedback = expert_model.improve_answer_with_feedback_and_critique(
            agieval_math_input_prompt, agieval_math_initial_output, epochs=2, iterations=10, is_math=is_math, examples=examples
        )

        cosine_sim = compute_similarity(expert_feedback, amateur_feedback)

        agieval_math_pred_answer = expert_model.generate_answer(refined_answer, is_math=is_math)
        agieval_math_pred_answer = agieval_math_pred_answer.replace(",", "").strip()
        agieval_math_pred_answer = re.sub(r"[^\d\.\-]", "", agieval_math_pred_answer)

        pred_answer = None
        try:
            pred_answer = int(agieval_math_pred_answer)
        except ValueError:
            try:
                pred_answer = float(agieval_math_pred_answer)
            except ValueError:
                pred_answer = None

        accuracy = int(ground_truth == pred_answer) if pred_answer is not None else 0

        similarities.append(cosine_sim)
        accuracies.append(accuracy)

        print(f"\n--- Task {i+1} ---")
        print(f"Prompt: {agieval_math_input_prompt}")
        print(f"x0 (Initial Output): {agieval_math_initial_output}")
        print(f"x1 (Refined Answer): {refined_answer}")
        print("Expert Feedback:", expert_feedback)
        print("Amateur Feedback:", amateur_feedback)
        print("Similarity:", round(cosine_sim, 3))
        print("Prediction:", pred_answer, "| Ground Truth:", ground_truth)
        print("Correct:", bool(accuracy))

        try:
            expert_tox = Detoxify("original").predict(expert_feedback)["toxicity"]
        except Exception as e:
            print(f"[Error] Expert Feedback Toxicity failed: {e}")
            expert_tox = None

        try:
            amateur_tox = Detoxify("original").predict(amateur_feedback)["toxicity"]
        except Exception as e:
            print(f"[Error] Amateur Feedback Toxicity failed: {e}")
            amateur_tox = None

        try:
            refined_tox = Detoxify("original").predict(refined_answer)["toxicity"]
        except Exception as e:
            print(f"[Error] Refined Answer Toxicity failed: {e}")
            refined_tox = None

        expert_feedback_toxicities.append(expert_tox)
        amateur_feedback_toxicities.append(amateur_tox)
        refined_answer_toxicities.append(refined_tox)

        result = {
            "question": agieval_math_input_prompt,
            "x0": agieval_math_initial_output,
            "expert_feedback": expert_feedback,
            "amateur_feedback": amateur_feedback,
            "similarity": cosine_sim,
            "x1": refined_answer,
            "pred": pred_answer,
            "label": ground_truth,
            "correct": bool(accuracy)
        }

        results.append(result)

        save_results_to_csv(
            result=result,
            features=["question", "x0", "expert_feedback", "amateur_feedback", "similarity", "x1", "pred", "label", "correct"],
            file_path=csv_path
        )

    plot_experiment_metrics(
        similarities,
        accuracies,
        save_path=f"{csv_dir}/{dataset_name}_similarity_accuracy_plot_main_gsm8k.png"
    )

    plot_toxicities(
        expert_feedback_toxicities,
        amateur_feedback_toxicities,
        refined_answer_toxicities,
        save_path=f"{csv_dir}/{dataset_name}_toxicity_plot_main_gsm8k.png"
    )

    print(f"\nExperiment complete. Results saved to: {csv_path}")
    return results

In [ ]:
agieval_experiment_result = run_experiment_agieval_math(sanitized_data, "agieval_math", expert_model, network)

## Experiment: Optimal Epoch & Iteration Search for Knowledge Distillation

This experiment is systematically varies training epochs and iterations to identify the configuration that yields the best trade-off between performance and computational cost in knowledge distillation. Multiple Loss Metrics are aggregated and normalzied for fair comparison. A Pareto Frontier analysis is then applied to determine the optimal settings, ensuring minimal trianing cost without sacrificing model quality.

In [ ]:
dataset_size = len(expert_datasets)
stopped_fractions = [i/100 for i in range(5, 11, 2)]
stopped_list = [int(dataset_size * frac) for frac in stopped_fractions]
epochs_list = [1, 3, 5]
features = ["epochs", "stopped", "final_avg_loss", "training_cost"]

In [ ]:
import gc
import torch

def getting_pareto_points(epochs_list, stopped_list, expert_datasets):
    results = []

    for epochs in epochs_list:
        for stopped in stopped_list:
            print(f"Running experiment: epochs={epochs}, stopped={stopped}")

            expert_model = ExpertFeedbackModel(teacher_tokenizer, teacher_model, teacher_model_namel)
            amateur_feedback_model = AmateurFeedbackModel(student_tokenizer, student_model, student_model_name)
            network = AmateurExpertFeedbackNetWork(amateur_feedback_model, expert_model, expert_datasets)
            expert_model.network = network

            result = network.knowledge_distillation_with_SFT(epochs=epochs, stopped=stopped)

            results.append({
                "epochs": epochs,
                "stopped": stopped,
                "training_cost": epochs * stopped,
                "final_avg_loss": result["final_avg_loss"]
            })

            del expert_model, amateur_feedback_model, network, result
            gc.collect()
            torch.cuda.empty_cache()


    pareto_points = []
    for r in results:
        dominated = False
        for other in results:
            if (other["training_cost"] <= r["training_cost"] and
                other["final_avg_loss"] <= r["final_avg_loss"] and
                (other["training_cost"] < r["training_cost"] or
                 other["final_avg_loss"] < r["final_avg_loss"])):
                dominated = True
                break
        if not dominated:
            pareto_points.append(r)

    return pareto_points

In [ ]:
pareto_frontier = getting_pareto_points(epochs_list, stopped_list, expert_datasets)

save_results_to_csv(pareto_frontier, features, "kd_optimal_epoch_and_iteration.csv")

converting_from_csv_to_json(features,
                            "kd_optimal_epoch_and_iteration.csv",
                            "kd_optimal_epoch_and_iteration.json")

NameError: name 'pareto_points' is not defined

In [ ]:
import matplotlib.pyplot as plt
import os

def plot_pareto_frontier(pareto_frontier, output_dir="kd_optimal_epoch_and_iteration_result", filename="kd_optimal_epoch_and_iteration_result.png"):
    pareto_frontier.sort(key=lambda x: x["training_cost"])

    x_vals = [p["training_cost"] for p in pareto_frontier]
    y_vals = [p["final_avg_loss"] for p in pareto_frontier]

    plt.figure(figsize=(10, 6))
    plt.plot(x_vals, y_vals, marker='o', linestyle='-')

    for p in pareto_frontier:
        label = f"e:{p['epochs']}, s:{p['stopped']}"
        plt.annotate(label, (p["training_cost"], p["final_avg_loss"]),
                     textcoords="offset points", xytext=(5, 5),
                     ha='left', fontsize=8)

    for p in pareto_frontier:
        print(f"Training config -> Epochs: {p['epochs']}, Stopped: {p['stopped']}, Cost: {p['training_cost']}, Loss: {p['final_avg_loss']}")

    plt.xlabel("Training Cost (epochs × num_iterations)")
    plt.ylabel("Final Avg Loss")
    plt.title("Pareto Frontier")
    plt.grid(True)

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, filename)

    plt.savefig(output_path, bbox_inches="tight")
    plt.show()

    print(f"Pareto frontier plot saved to: {output_path}")

In [ ]:
plot_pareto_frontier(pareto_frontier)

## Experiment: Performance Threshold for Student Against Teacher Performnace

This experiment evaluates different student performance thresholds to determine the most cost-effective stopping point for knowledge distillation. Using the *What's That Book* dataset's test split, the system runs a fixed nummber of epochs and iterations for each threshold, generating combined expert-amateur feedback and tracking multi-metric losses.

For each threshold:
  * **Training Cost** is calculated as `iterations x epochs`

  * **Final average** is computed from all evaluated samples.


  * Results are compared via **Pareto frontier analysis** to identify thresholds that offer the best trade-off between accuracy and cost

  * The best point is chosen by minimizing the product of training cost and loss.

In [ ]:
from datasets import load_dataset

wtb_dataset = load_dataset("nlpkevinl/whatsthatbook")

wtb_train_dataset = wtb_dataset["train"]
wtb_test_dataset = wtb_dataset["test"]

In [ ]:
import numpy as np

epochs = 1
iterations = 10

performance_threshold = list(np.arange(0.8, 1.01, 0.05))
features = ["threshold", "training_cost", "final_avg_loss"]

In [ ]:
import gc
import torch

def run_experiment_and_get_pareto(performance_thresholds, datasets, expert_datasets, epochs, iterations):
    results_per_threshold = []

    for performance in performance_thresholds:
        print(f"\n=== Running Experiment with Loss Threshold = {performance} ===")
        all_losses = []

        expert_model = ExpertFeedbackModel(teacher_tokenizer, teacher_model, teacher_model_name)
        amateur_feedback_model = AmateurFeedbackModel(student_tokenizer, student_model, student_model_name)
        network = AmateurExpertFeedbackNetWork(amateur_feedback_model, expert_model, expert_datasets)
        expert_model.network = network

        for i, sample in enumerate(datasets):
            if i >= 2:
                break

            prompt = sample["question"]
            answer = sample["answers"]
            result = network.generate_combined_feedback(prompt, answer, epochs, iterations, performance)
            all_losses.extend(result["loss_vector"])

        avg_loss = sum(all_losses) / len(all_losses) if all_losses else float('inf')
        total_cost = iterations * epochs

        results_per_threshold.append({
            "threshold": performance,
            "training_cost": total_cost,
            "final_avg_loss": avg_loss,
        })

        del expert_model, amateur_feedback_model, network
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_per_threshold.sort(key=lambda x: x["training_cost"])
    pareto_frontier = []
    for candidate in results_per_threshold:
        if not any(
            other["training_cost"] < candidate["training_cost"] and
            other["final_avg_loss"] < candidate["final_avg_loss"]
            for other in results_per_threshold
        ):
            pareto_frontier.append(candidate)

    best_point = min(
        pareto_frontier,
        key=lambda p: p["training_cost"] * p["final_avg_loss"]
    )

    return pareto_frontier, best_point

In [ ]:
pareto_frontier, best_point = run_experiment_and_get_pareto(performance_threshold, wtb_test_dataset, expert_datasets, epochs, iterations)
print("\n=== Pareto Frontier Results ===")
for p in pareto_frontier:
  print(f"Threshold: {p['threshold']}, Training Cost: {p['training_cost']}, Final Avg Loss: {p['final_avg_loss']:.4f}")

save_results_to_csv(pareto_frontier, features, "kd_threshold_experiment.csv")

converting_from_csv_to_json(features,
                            "kd_threshold.csv",
                            "kd_threshold_experiment.json")

In [ ]:
import matplotlib.pyplot as plt
import os

def save_plot_to_file(fig, output_dir, filename):
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, bbox_inches="tight")
    print(f"Plot saved to {save_path}")
    plt.close(fig)

def to_float(x):
    return float(x.cpu()) if hasattr(x, 'cpu') else float(x)

def plot_pareto_frontier(pareto_frontier, best_point, output_dir="kd_opt"):
    training_costs = [to_float(p["training_cost"]) for p in pareto_frontier]
    final_avg_losses = [to_float(p["final_avg_loss"]) for p in pareto_frontier]

    best_cost = to_float(best_point["training_cost"])
    best_loss = to_float(best_point["final_avg_loss"])

    fig = plt.figure(figsize=(10, 6))
    plt.plot(training_costs, final_avg_losses, marker='o', label="Pareto Points")

    for p in pareto_frontier:
        label = f"e:{epochs}, s:{iterations}, threshold: {p['threshold']:.2f}"
        plt.annotate(label, (to_float(p["training_cost"]), to_float(p["final_avg_loss"])),
                     textcoords="offset points", xytext=(5, 5),
                     ha='left', fontsize=8)

    plt.scatter(best_cost, best_loss, color='red', label="Best Point", s=100, zorder=5)
    plt.xlabel("Training Cost (epochs × iterations)")
    plt.ylabel("Final Avg Loss")
    plt.title("Pareto Frontier of KD Training")
    plt.legend()
    plt.grid(True)

    save_plot_to_file(fig, output_dir, "kd_performance_threshold.png")

In [ ]:
plot_pareto_frontier(pareto_frontier, best_point)